# TORRS — MA2C Traffic Optimisation for Malta
### Complete codebase notebook

This notebook bundles every source file needed to run the project so you can
read, annotate, and experiment in one place.

**Run order during a real experiment:**
1. *(once)* `fix_detectors.py` — regenerate `exp.add.xml` from the network file
2. `main.py` with `option = 'train'` — trains the MA2C model
3. `main.py` with `option = 'greedy'` — baseline comparison
4. `main.py` with `option = 'evaluate'` — full evaluation run
5. Results section at the bottom — load CSVs and plot

> **Note:** cells that start SUMO (`main.py`, `envs/`) require SUMO to be
> installed and `Malta/data/malta.net.xml` to be present.
> All other cells (agents, utils, config) can be read / imported standalone.


## 1. Prerequisites & Installation

The project needs Python ≥ 3.10, TensorFlow 2.x (CPU mode), SUMO ≥ 1.14,
and a few data/plotting libraries.  Run the cell below once to install the
Python packages (SUMO itself must be installed separately via its installer).


In [1]:
import os, sys
NOTEBOOK_DIR = r"C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook"
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())
print("utils.py exists:", os.path.exists("utils.py"))

cwd: C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook
utils.py exists: True


In [2]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'      
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['TF_NUM_INTRAOP_THREADS'] = '4'
os.environ['TF_NUM_INTEROP_THREADS'] = '2'

import sys
NOTEBOOK_DIR = r"C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook"
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd(), "| oneDNN disabled")

cwd: C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook | oneDNN disabled


In [3]:
import subprocess, sys

# All packages the notebook needs
packages = [
    ("tensorflow",  "tensorflow==2.18.0"),
    ("numpy",       "numpy"),
    ("pandas",      "pandas"),
    ("scipy",       "scipy"),
    ("seaborn",     "seaborn"),
    ("matplotlib",  "matplotlib"),
]

print("Checking / installing packages...")
all_ok = True
for import_name, install_name in packages:
    try:
        __import__(import_name)
        print(f"  OK        {import_name}")
    except ImportError:
        print(f"  Installing {install_name} ...")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", install_name],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"  Installed  {import_name}")
        else:
            print(f"  FAILED     {import_name}")
            print(result.stderr[-400:])
            all_ok = False

print()
if all_ok:
    print("All packages ready.")
else:
    print("One or more installs failed -- check the error above.")
print("Tip: TensorFlow requires Python 3.9-3.12 (not 3.13).")


Checking / installing packages...
  OK        tensorflow
  OK        numpy
  OK        pandas
  OK        scipy
  OK        seaborn
  OK        matplotlib

All packages ready.
Tip: TensorFlow requires Python 3.9-3.12 (not 3.13).


## 1b. Project Path Setup

Tells Python where to find the project modules (, , , ).
Run this cell before any other code cell.
Change  below if you put the folder somewhere else.

In [4]:
import sys, os

PROJECT_ROOT = r"C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())
print("Folders found:", [f for f in os.listdir(PROJECT_ROOT) if os.path.isdir(os.path.join(PROJECT_ROOT, f))])


Working directory: C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook
Folders found: ['.claude', 'agents', 'config', 'envs', 'Malta', '__pycache__']


## 2. GPU Check (`gpu.py`)

Quick sanity-check: prints the TensorFlow version and any visible GPU devices.
The training loop forces CPU-only by default (see `main.py`) because the
per-agent batch size is too small to benefit from a GPU.


In [5]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
print(tf.__version__)

[]
2.21.0


## 3. Configuration (`config/config_ma2c_Malta.ini`)

All hyper-parameters live here so you can tune without touching any Python
file.  Key values to know:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `batch_size` | 144 | Steps per A2C update (= 1 SUMO minute at 5 s intervals) |
| `total_step` | 10 000 | Total training steps (~7 episodes) |
| `gamma` | 0.99 | Reward discount factor |
| `lr_init` | 2.5e-4 | Adam learning rate (constant) |
| `entropy_coef_s` | 0.05 | Signal entropy bonus (encourages exploration) |
| `entropy_coef_r` | 0.01 | Route entropy bonus |
| `num_lstm` | 128 | LSTM hidden units per agent |
| `coop_gamma_ss` | 0.3 | Signal→signal neighbourhood discount |
| `coop_gamma_sr` | 0.3 | Signal→route neighbourhood discount |


In [6]:
# Display the configuration file
with open("config/config_ma2c_Malta.ini") as f:
    print(f.read())


[MODEL_CONFIG]
adam_beta1=0.9
adam_beta2=0.999
adam_epsilon=1e-08
entropy_coef_s = 0.05
entropy_coef_r = 0.01

max_grad_norm = 30
gamma = 0.99
lr_init = 2.5e-4
lr_decay = constant
; route
r_num_fw=20
r_num_ft=70
r_num_fp=10
;signal
s_num_fw=140
s_num_ft=60
s_num_fp=70
; signal_route
num_fw_s = 140
num_ft_s = 60
num_fo_s = 50
num_fp_s = 70
num_fw_r = 20
num_ft_r = 70
num_fo_r = 200
num_fp_r =50
num_fs=128
num_lstm = 128

batch_size = 144

[TRAIN_CONFIG]
total_step = 10000
test_interval = 1440
log_interval = 720

[ENV_CONFIG]
clip_wave = 2.0
clip_wait = 2.0
clip_arrival = 2.0
clip_occupancy= 2.0
norm_wave = 2
norm_wait = 8
norm_arrival = 16
norm_occupancy = 7

clip_reward = 6.0
norm_reward_signal = 400
norm_reward_route = 100
norm_reward_sr_signal = 400
norm_reward_sr_route = 100

CAV_rate = 1
control_interval_sec = 5
yellow_interval_sec = 5
episode_length_sec = 3600

; agent is greedy, iqll, iqld, ia2c, ma2c, a2c.
agent = ma2c
; coop discount is used to discount the neighbors' impact
co

## 4. Fix Detectors (`fix_detectors.py`)

One-off utility: reads `Malta/data/malta.net.xml` and writes
`Malta/data/exp.add.xml` containing an E2 lane-area detector on every
non-internal lane.  Run this once if the additional file is missing or the
network changes.  The proper version used during training is generated inside
`Malta/build_file.py → output_additional()` which only adds detectors for
lanes feeding traffic-light junctions (much smaller file).


In [7]:
import xml.etree.ElementTree as ET

tree = ET.parse('Malta/data/malta.net.xml')
root = tree.getroot()

lines = ['<additional>']
for edge in root.findall('edge'):
    edge_id = edge.get('id')
    if edge_id is None or edge_id.startswith(':'):
        continue
    for lane in edge.findall('lane'):
        lane_id = lane.get('id')
        length = float(lane.get('length', 100))
        det_length = min(length, 100)
        lines.append(f'    <laneAreaDetector id="{lane_id}" lane="{lane_id}" pos="0" length="{det_length:.2f}" freq="1" file="ild_out.xml"/>')
lines.append('</additional>')

with open('Malta/data/exp.add.xml', 'w') as f:
    f.write('\n'.join(lines))
print('Done! Written', len(lines)-2, 'detectors')

Done! Written 120646 detectors


## 5. Malta Network Builder (`Malta/build_file.py`)

Reads `malta.net.xml` directly (no hand-crafted CSV tables) and produces:

* **`get_nodes()`** — list of TL junctions (≥2 phases) and non-TL junctions
* **`get_phases()`** — maps each TL junction to its allowed phase states
* **`get_routes()`** — 4 route-choice corridors × 2 alternatives; invalid
  routes are auto-repaired via BFS over the connection graph
* **`get_flows()`** — vehicles/hour per corridor (Malta peak-hour calibration)
* **`gen_rou_file()`** — called by the env each episode to write the SUMO
  `.rou.xml` and `.sumocfg` files with randomised seed demand


In [8]:
# -*- coding: utf-8 -*-
"""
Malta/build_file.py
Replaces Sioux/data/build_file.py for the Malta road network.
Reads directly from malta.net.xml instead of CSV files.
"""

import os
import xml.etree.ElementTree as ET
import numpy as np
from collections import defaultdict, deque

DATA_DIR = os.path.join(os.getcwd(), 'Malta', 'data')  # notebook-safe (no __file__)
NET_FILE  = os.path.join(DATA_DIR, 'malta.net.xml')

MAX_CAR_NUM  = 30
SPEED_LIMIT  = 50
time_length  = 3600


# ── helpers ───────────────────────────────────────────────────────────────────

# Process-wide cache of the parsed net + derived structures.
# Parsing malta.net.xml is expensive (~2 MB, 400k+ edges), so we keep one copy.
_NET_CACHE = {}


def _parse_net():
    if 'root' not in _NET_CACHE:
        _NET_CACHE['root'] = ET.parse(NET_FILE).getroot()
    return _NET_CACHE['root']


def _lane_allows_passenger(lane_elem, edge_disallow, edge_allow):
    allow = lane_elem.attrib.get('allow', edge_allow)
    disallow = lane_elem.attrib.get('disallow', edge_disallow)
    if allow:
        return 'passenger' in allow.split()
    if disallow:
        return 'passenger' not in disallow.split()
    return True  # SUMO default: passenger allowed


def _build_net_index():
    """One-shot scan of malta.net.xml: identify passenger-allowed edges, build
    an edge connectivity graph from <connection> elements, and collect
    traffic-light junctions. Cached for the life of the process."""
    if _NET_CACHE.get('index_built'):
        return
    root = _parse_net()

    passenger_edges = set()
    edge_from = {}
    edge_to = {}
    for edge in root.findall('edge'):
        if edge.attrib.get('function') == 'internal':
            continue
        eid = edge.attrib.get('id', '')
        if not eid or eid.startswith(':'):
            continue
        e_disallow = edge.attrib.get('disallow', '')
        e_allow = edge.attrib.get('allow', '')
        lanes = edge.findall('lane')
        if not lanes:
            continue
        if any(_lane_allows_passenger(l, e_disallow, e_allow) for l in lanes):
            passenger_edges.add(eid)
            edge_from[eid] = edge.attrib.get('from', '')
            edge_to[eid] = edge.attrib.get('to', '')

    edge_graph = defaultdict(set)
    for conn in root.findall('connection'):
        fr = conn.attrib.get('from', '')
        to = conn.attrib.get('to', '')
        if fr in passenger_edges and to in passenger_edges:
            edge_graph[fr].add(to)

    tl_junctions = set()
    for j in root.findall('junction'):
        if j.attrib.get('type') == 'traffic_light':
            tl_junctions.add(j.attrib.get('id', ''))

    _NET_CACHE['passenger_edges'] = passenger_edges
    _NET_CACHE['edge_from'] = edge_from
    _NET_CACHE['edge_to'] = edge_to
    _NET_CACHE['edge_graph'] = edge_graph
    _NET_CACHE['tl_junctions'] = tl_junctions
    _NET_CACHE['index_built'] = True


def _route_is_valid(edge_seq):
    """A route is valid if every edge allows passenger and consecutive edges
    have a SUMO connection between them."""
    if not edge_seq:
        return False
    _build_net_index()
    passenger_edges = _NET_CACHE['passenger_edges']
    edge_graph = _NET_CACHE['edge_graph']
    for e in edge_seq:
        if e not in passenger_edges:
            return False
    for i in range(len(edge_seq) - 1):
        if edge_seq[i + 1] not in edge_graph.get(edge_seq[i], set()):
            return False
    return True


def _bfs_path(src, max_depth=15, want_tl_count=2):
    """BFS from src; return the first path with >= want_tl_count TL crossings
    and length >= 4. Returns None if no such path exists."""
    _build_net_index()
    edge_graph = _NET_CACHE['edge_graph']
    edge_to = _NET_CACHE['edge_to']
    tl_junctions = _NET_CACHE['tl_junctions']
    visited = {src: None}
    queue = deque([(src, 0, 0)])
    while queue:
        cur, depth, tl_count = queue.popleft()
        if tl_count >= want_tl_count and depth >= 4:
            path = []
            x = cur
            while x is not None:
                path.append(x)
                x = visited[x]
            return list(reversed(path))
        if depth >= max_depth:
            continue
        for nxt in edge_graph.get(cur, ()):
            if nxt in visited:
                continue
            visited[nxt] = cur
            new_tl = tl_count + (1 if edge_to.get(nxt) in tl_junctions else 0)
            queue.append((nxt, depth + 1, new_tl))
    return None


def _validate_or_fix_routes(routes):
    """For every (corridor, alternative) edge sequence, keep it if valid;
    otherwise replace it with a BFS-discovered passenger-allowed path that
    crosses at least 2 TL junctions. Each corridor is guaranteed to end
    with exactly 2 alternatives (env requires this)."""
    _build_net_index()
    passenger_edges = _NET_CACHE['passenger_edges']
    sources_pool = sorted(passenger_edges)
    rng = np.random.default_rng(123)

    fixed = []
    for i, route_set in enumerate(routes):
        new_set = []
        for r in route_set:
            if _route_is_valid(r):
                new_set.append(r)
                continue
            # try to extend the original first edge if it's at least passenger-allowed
            seed_src = r[0] if (r and r[0] in passenger_edges) else None
            new_path = _bfs_path(seed_src) if seed_src else None
            if new_path is None:
                # last resort: pick random passenger-allowed sources
                for _ in range(50):
                    cand = sources_pool[rng.integers(0, len(sources_pool))]
                    new_path = _bfs_path(cand)
                    if new_path:
                        break
            if new_path is None:
                # absolute last resort: a single passenger edge — SUMO will
                # accept it (single-edge route) even if it doesn't cross a TL
                new_path = [sources_pool[rng.integers(0, len(sources_pool))]]
            print('[Malta route fix] corridor %d alt %d: replaced %r with %d-edge path'
                  % (i + 1, len(new_set) + 1, r, len(new_path)))
            new_set.append(new_path)
        # env requires exactly 2 alternatives per corridor
        while len(new_set) < 2:
            new_set.append(new_set[0])
        fixed.append(new_set[:2])
    return fixed


def write_file(path, content):
    with open(path, 'w') as f:
        f.write(content)


# ── required by Malta_env.py ──────────────────────────────────────────────────

def get_nodes(build=False):
    root = _parse_net()

    # Get valid TL IDs — must have >= 2 phases
    valid_tl_ids = set()
    for tl in root.findall('tlLogic'):
        tl_id  = tl.attrib.get('id')
        states = [p.attrib['state'] for p in tl.findall('phase') if 'state' in p.attrib]
        if tl_id and len(states) >= 2:
            valid_tl_ids.add(tl_id)

    tl_nodes, non_tl_nodes = [], []
    for j in root.findall('junction'):
        jid   = j.attrib.get('id', '')
        jtype = j.attrib.get('type', '')
        if jid.startswith(':'):
            continue
        if jtype == 'traffic_light':
            if jid in valid_tl_ids:
                tl_nodes.append(jid)
            else:
                # treat excluded TL junctions as non-TL
                non_tl_nodes.append(jid)
        elif jtype in ('priority', 'right_before_left', 'unregulated'):
            non_tl_nodes.append(jid)

    return tl_nodes, non_tl_nodes



def get_phases():
    root = _parse_net()
    phases = {}
    phase_node_map = {}
    for tl in root.findall('tlLogic'):
        tl_id  = tl.attrib.get('id')
        states = [p.attrib['state'] for p in tl.findall('phase') if 'state' in p.attrib]
        # SKIP junctions with only 1 phase — model needs at least 2 actions
        if tl_id and len(states) >= 2:
            phases[tl_id]          = states
            phase_node_map[tl_id]  = tl_id
    return phases, phase_node_map


def get_routes():
    """
    Route-choice alternatives for Malta corridors using real SUMO edge IDs.

    Corridor 1 — Route choice at junction 6484114392 (two alternatives):
      Route A: via 1657799943 and 301487637 (longer alternative)
      Route B: direct to 6484114393

    Corridors 2-4 — Bidirectional arterial pairs
    """
    routes = [
        # Corridor 1: main route choice point
        # Route A: 54444410#2 -> 1264291443 -> 33698427#1
        # Route B: 824060363#1 (direct)
        [
            ['54444410#2', '1264291443', '33698427#1'],
            ['824060363#1'],
        ],

        # Corridor 2: bidirectional arterial
        [
            ['-1456229447#1'],
            ['1456229447#1'],
        ],

        # Corridor 3: bidirectional arterial
        [
            ['-27446664#1'],
            ['27446664#1'],
        ],

        # Corridor 4: bidirectional arterial
        [
            ['-743322159'],
            ['743322159'],
        ],
    ]

    # Auto-validate and repair: any route whose edges aren't connected in the
    # network (or aren't passenger-allowed) gets replaced with a BFS-discovered
    # path. This stops SUMO from erroring out on "no valid route" and from
    # silently dropping flows on disallowed edges.
    routes = _validate_or_fix_routes(routes)
    rc_num = sum(len(r) - 1 for r in routes)
    return routes, rc_num


def relevant_ss(build=False):
    """
    Return a dict mapping each TL node to its directly connected TL neighbours.
    Built automatically from the edges in malta.net.xml.
    """
    root = _parse_net()
    tl_nodes, _ = get_nodes()
    tl_set = set(tl_nodes)

    relevant_ss_map = {n: [] for n in tl_nodes}
    for edge in root.findall('edge'):
        if edge.attrib.get('function') == 'internal':
            continue
        frm = edge.attrib.get('from', '')
        to  = edge.attrib.get('to', '')
        if frm in tl_set and to in tl_set:
            if to not in relevant_ss_map[frm]:
                relevant_ss_map[frm].append(to)
    return relevant_ss_map


def get_flows(build=False):
    """
    Vehicles/hour per corridor calibrated to Malta peak-hour conditions.
    Index matches corridor order in get_routes().
    Corridor 1: main route-choice point (high demand, Msida-Valletta type)
    Corridors 2-4: bidirectional arterial flows
    """
    return [600, 300, 400, 350]


# ── route/config file generation ─────────────────────────────────────────────

def _get_edge_ids_for_route(edge_sequence):
    """Routes are already edge IDs — return them directly."""
    return edge_sequence


def output_flows(demand_flows, seed=None, load=False):
    routes, _ = get_routes()
    demand_begin = 0
    demand_end   = 3600
    demand_gap   = 360

    ratios = [
        [0.3, 0.5, 0.4, 0.3, 0.8, 0.9, 1.0, 0.9, 0.7, 0.2],
        [0.3, 0.2, 0.4, 0.3, 0.5, 0.5, 0.8, 0.8, 0.9, 1.0],
        [0.2, 0.3, 0.5, 0.4, 0.7, 0.8, 1.0, 0.9, 0.6, 0.3],
        [0.4, 0.5, 0.6, 0.5, 0.8, 0.9, 1.0, 0.8, 0.5, 0.2],
    ]

    if seed is not None:
        np.random.seed(seed)

    str_flows = '<routes>\n'
    str_flows += '    <vType id="car" accel="2.6" decel="4.5" sigma="0.5" length="5" maxSpeed="50"/>\n'

    # Define route alternatives — edges already provided directly
    for i, route_set in enumerate(routes):
        for k, edge_seq in enumerate(route_set):
            edges = ' '.join(edge_seq)
            str_flows += '    <route id="route_%d_%d" edges="%s"/>\n' % (i+1, k+1, edges)

    # Generate flows per time interval
    for j in range(10):
        t_begin = j * demand_gap
        t_end   = (j + 1) * demand_gap
        for i, route_set in enumerate(routes):
            flow_val = ratios[i % len(ratios)][j] * demand_flows[i % len(demand_flows)]
            if load:
                flow_val = int(flow_val * load)
            str_flows += (
                '    <flow id="flow_%d_%d" route="route_%d_1" '
                'begin="%d" end="%d" vehsPerHour="%.1f" type="car"/>\n'
                % (i+1, j+1, i+1, t_begin, t_end, flow_val)
            )

    str_flows += '</routes>\n'
    return str_flows


def output_config(thread=None, add_file=None):
    if thread is None:
        rou_file = 'exp.rou.xml'
    else:
        rou_file = 'exp_%d.rou.xml' % int(thread)

    add_line = ''
    if add_file:
        add_line = '        <additional-files value="%s"/>\n' % add_file

    cfg = (
        '<configuration>\n'
        '    <input>\n'
        '        <net-file value="malta.net.xml"/>\n'
        '        <route-files value="%s"/>\n'
        '%s'
        '    </input>\n'
        '    <time>\n'
        '        <begin value="0"/>\n'
        '        <end value="%d"/>\n'
        '    </time>\n'
        '    <processing>\n'
        '        <time-to-teleport value="-1"/>\n'
        '    </processing>\n'
        '</configuration>\n'
    ) % (rou_file, add_line, time_length)
    return cfg


def output_additional(data_path):
    """
    Generate E2 lane-area detectors only on lanes that lead INTO traffic-light
    junctions. The env's _init_agents only ever queries detectors that match
    trafficlight.getControlledLanes(), so emitting one per lane in the entire
    network (~120k detectors) was massive wasted work at SUMO startup.

    Also: dropped freq=1 + per-second file output (was ~430M lines/episode).
    The env reads detector state via TraCI getLastStep* calls, which work
    independently of the aggregation period.
    """
    root = _parse_net()

    tl_junctions = set()
    for j in root.findall('junction'):
        if j.attrib.get('type') == 'traffic_light':
            tl_junctions.add(j.attrib.get('id', ''))

    str_add = '<additional>\n'
    count = 0
    for edge in root.findall('edge'):
        if edge.attrib.get('function') == 'internal':
            continue
        eid = edge.attrib.get('id', '')
        if not eid or eid.startswith(':'):
            continue
        # Only lanes feeding a TL junction matter for the agents
        if edge.attrib.get('to', '') not in tl_junctions:
            continue
        for lane in edge.findall('lane'):
            lane_id = lane.attrib.get('id', '')
            lane_len = float(lane.attrib.get('length', 100))
            det_len = min(lane_len - 0.1, 100)
            if det_len <= 0:
                det_len = 1.0
            # SUMO requires both `file` and a period; we set period to the
            # full episode length so only ~one row per detector is ever
            # written (442 rows total, negligible). Per-step queries via
            # TraCI getLastStep* still work regardless of period.
            str_add += (
                '    <laneAreaDetector id="%s_0" lane="%s" '
                'pos="0" length="%.2f" period="3600" file="ild_out.xml"/>\n'
                % (lane_id, lane_id, det_len)
            )
            count += 1
    str_add += '</additional>\n'
    print('[Malta detectors] emitted %d laneAreaDetectors (TL-incoming lanes only)' % count)
    return str_add


def gen_rou_file(path, demand_flows, seed=None, thread=None, load=False):
    if thread is None:
        flow_file = 'exp.rou.xml'
    else:
        flow_file = 'exp_%d.rou.xml' % int(thread)

    write_file(path + flow_file, output_flows(demand_flows, seed=seed, load=load))

    # generate detector additional file
    add_file = path + 'exp.add.xml'
    write_file(add_file, output_additional(path))

    sumocfg_file = path + ('exp_%d.sumocfg' % thread)
    write_file(sumocfg_file, output_config(thread=thread, add_file='exp.add.xml'))
    return sumocfg_file

## 6. Base Traffic Simulator (`envs/env.py`)

Abstract SUMO environment that all network-specific envs inherit from.

**Key classes:**

| Class | Purpose |
|-------|---------|
| `Node` | One TL junction agent — holds lanes, detectors, state, reward |
| `Rc` | One routing corridor agent — holds edge maps, detector, state |
| `PhaseSet / PhaseMap` | Encodes allowed traffic-light phase strings |
| `RouteSet / RouteMap` | Encodes route alternatives per corridor |
| `TrafficSimulator` | Abstract base — launches SUMO via TraCI, steps the sim, computes states and cooperative rewards |

**Reward structure:**
Signal reward = −(queue length × interval) − (max vehicle wait).
Route reward = number of target vehicles that arrived at the corridor end.
Cooperative neighbours' rewards are blended in via `coop_gamma_*` discounts.


In [9]:
import logging
from platform import node
import numpy as np
import pandas as pd
import subprocess
from sumolib import checkBinary
import time
import traci
import random
import xml.etree.cElementTree as ET
import os

DEFAULT_PORT = 27500
SEC_IN_MS = 1000

REALNET_REWARD_NORM = 20


class RouteSet:
    def __init__(self, routes):
        self.num_route = len(routes)
        self.routes = routes


class PhaseSet:
    def __init__(self, phases):
        self.num_phase = len(phases)
        self.num_lane = len(phases[0])
        self.phases = phases

    @staticmethod
    def _get_phase_lanes(phase, signal='r'):
        phase_lanes = []
        for i, l in enumerate(phase):
            if l == signal:
                phase_lanes.append(i)
        return phase_lanes

    def _init_phase_set(self):
        self.red_lanes = []
        for phase in self.phases:
            self.red_lanes.append(self._get_phase_lanes(phase))


class RouteMap:
    def __init__(self):
        self.routes = {}

    def get_route(self, rc_id, action):
        return self.routes[rc_id].routes

    def get_route_num(self, route_id):
        return self.routes[route_id].num_route


class PhaseMap:
    def __init__(self):
        self.phases = {}

    def get_phase(self, phase_id, action):
        return self.phases[phase_id].phases[int(action)]

    def get_phase_num(self, phase_id):
        return self.phases[phase_id].num_phase

    def get_lane_num(self, phase_id):
        return self.phases[phase_id].num_lane

    def get_red_lanes(self, phase_id, action):
        return self.phases[phase_id].red_lanes[int(action)]


class Node:
    def __init__(self, name, relevant_ss=[], relevant_sr={}, control=False):
        self.control = control
        self.lanes_in = []
        self.ilds_in = []
        self.fingerprint = []
        self.name = name
        self.relevant_ss = relevant_ss
        self.relevant_sr = relevant_sr
        self.num_state = 0
        self.num_fingerprint = 0
        self.wave_state = []
        self.wait_state = []
        self.reward = 0
        self.wait = 0
        self.vehwait = 0
        self.phase_id = -1
        self.n_a = 0
        self.prev_action = -1


class Rc:
    def __init__(self, name, f_id=None, relevant_rr=[], relevant_rs={}, up_edge=None, down_edge={},
                 forward_route=[], det=[], control=False):
        self.control = control
        self.fingerprint = []
        self.up_edge = 0
        self.name = name
        self.f_id = f_id
        self.det = det
        self.relevant_rr = relevant_rr
        self.relevant_rs = relevant_rs
        self.down_edge = down_edge
        self.up_edge = up_edge
        self.forward_route = forward_route
        self.num_state_arrival = 0
        self.num_state_occupancy = 0
        self.num_fingerprint = 0
        self.arrived_vehicles = 0
        self.reward = 0
        self.wait = 0
        self.arrived = 0
        self.arrival_state = []
        self.occupancy_state = []
        self.route_id = -1
        self.n_a = 0
        self.prev_action = -1


class TrafficSimulator:
    def __init__(self, config, problem, output_path, is_record, port=0):
        self.name = config.get('scenario')
        self.seed = config.getint('seed')
        self.control_interval_sec = config.getint('control_interval_sec')
        self.episode_length_sec = config.getint('episode_length_sec')
        self.T = np.ceil(self.episode_length_sec / self.control_interval_sec)
        self.CAV_rate = config.getfloat('CAV_rate')
        self.port = DEFAULT_PORT + port
        self.sim_thread = port
        self.data_path = config.get('data_path')
        self.agent = config.get('agent')
        self.problem = problem
        self.coop_gamma_ss = config.getfloat('coop_gamma_ss')
        self.coop_gamma_sr = config.getfloat('coop_gamma_sr')
        self.coop_gamma_rr = config.getfloat('coop_gamma_rr')
        self.coop_gamma_rs = config.getfloat('coop_gamma_rs')
        self.cur_episode = 0
        self.norms = {'wave': config.getfloat('norm_wave'),
                      'wait': config.getfloat('norm_wait'),
                      'arrival': config.getfloat('norm_arrival'),
                      'occupancy': config.getfloat('norm_occupancy')}
        if self.problem == 'signal':
            self.norms['reward_signal'] = config.getfloat('norm_reward_signal')
        elif self.problem == 'route':
            self.norms['reward_route'] = config.getfloat('norm_reward_route')
        elif self.problem == 'signal_route':
            self.norms['reward_signal'] = config.getfloat('norm_reward_sr_signal')
            self.norms['reward_route'] = config.getfloat('norm_reward_sr_route')
        self.clips = {'wave': config.getfloat('clip_wave'),
                      'wait': config.getfloat('clip_wait'),
                      'arrival': config.getfloat('clip_arrival'),
                      'occupancy': config.getfloat('clip_occupancy'),
                      'reward': config.getfloat('clip_reward')}
        self.coef_rreward = config.getfloat('coef_rreward')
        self.coef_vehwait = config.getfloat('coef_vehwait')
        self.train_mode = True
        test_seeds = config.get('test_seeds').split(',')
        test_seeds = [int(s) for s in test_seeds]
        self._init_map()
        self.init_data(is_record, output_path)
        self.init_test_seeds(test_seeds)
        self._init_sim(self.seed)
        self._init_agents()
        self.terminate()

    def _debug_traffic_step(self):
        for node_name in self.node_names:
            node = self.nodes[node_name]
            phase = self.sim.trafficlight.getRedYellowGreenState(self.node_names[0])
            cur_traffic = {'episode': self.cur_episode,
                           'time_sec': self.cur_sec,
                           'node': node_name,
                           'action': node.prev_action,
                           'phase': phase}
            for i, ild in enumerate(node.ilds_in):
                cur_name = 'lane%d_' % i
                cur_traffic[cur_name + 'queue'] = self.sim.lane.getLastStepHaltingNumber(ild)
                cur_traffic[cur_name + 'flow'] = self.sim.lane.getLastStepVehicleNumber(ild)
            self.traffic_data.append(cur_traffic)

    def _get_node_phase(self, action, node_name):
        node = self.nodes[node_name]
        cur_phase = self.phase_map.get_phase(node.phase_id, action)
        prev_action = node.prev_action
        node.prev_action = action
        if (prev_action < 0) or (action == prev_action):
            return cur_phase
        prev_phase = self.phase_map.get_phase(node.phase_id, prev_action)
        switch_reds = []
        switch_greens = []
        for i, (p0, p1) in enumerate(zip(prev_phase, cur_phase)):
            if (p0 in 'Gg') and (p1 == 'r'):
                switch_reds.append(i)
            elif (p0 in 'r') and (p1 in 'Gg'):
                switch_greens.append(i)
        if not len(switch_reds):
            return cur_phase
        yellow_phase = list(cur_phase)
        for i in switch_reds:
            yellow_phase[i] = 'y'
        for i in switch_greens:
            yellow_phase[i] = 'r'
        return ''.join(yellow_phase)

    def _get_node_phase_id(self, node_name):
        raise NotImplementedError()

    def _get_rc_state_num(self):
        raise NotImplementedError()

    def _get_rc_route_num(self, rc_name):
        raise NotImplementedError()

    def _get_rc_names(self):
        raise NotImplementedError()

    def _get_node_names(self):
        raise NotImplementedError()

    def _get_node_state_num(self, node):
        if len(node.lanes_in) != self.phase_map.get_lane_num(node.phase_id):
            best = self._find_matching_phase(len(node.lanes_in))
            if best:
                node.phase_id = best
        return len(node.ilds_in)

    def _get_reward(self, get_parameter=False):
        if get_parameter:
            norm_sreward = []
            norm_rreward = []
            norm_rarrived = []
            norm_swait = []
            norm_svehwait = []
        rewards_signal = []
        rewards_route = []
        for node_name in self.node_names:
            node = self.nodes[node_name]
            reward = node.reward
            if get_parameter:
                norm_sreward.append(reward)
                norm_swait.append(node.wait)
                norm_svehwait.append(node.vehwait)
            rewards_signal.append(reward)
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            reward = rc.reward
            if get_parameter:
                norm_rreward.append(reward)
                norm_rarrived.append(rc.arrived)
            rewards_route.append(self.coef_rreward * reward)
        if get_parameter:
            sreward_ave = np.mean(norm_sreward)
            rreward_ave = np.mean(norm_rreward)
            rarrived_ave = np.mean(norm_rarrived)
            swait_ave = np.mean(norm_swait)
            svehwait_ave = np.mean(norm_svehwait)
            reward_norm_ave = {'sreward': sreward_ave, 'rreward': rreward_ave,
                               'rarrived_ave': rarrived_ave, 'swait_ave': swait_ave,
                               'svehwait_ave': svehwait_ave}
            self.problem = 'signal'
            signal_rewards_ave = np.mean(self._local_reward_step(np.array(rewards_signal)))
            self.problem = 'route'
            route_rewards_ave = np.mean(self._local_reward_step(np.array(rewards_route)))
            self.problem = 'signal_route'
            rewards = self._local_reward_step(np.array(rewards_signal + rewards_route))
            sr_signal_rewards_ave = np.mean(rewards[:len(self.node_names)])
            sr_route_rewards_ave = np.mean(rewards[len(self.node_names):])
            total_rewards_ave = {'signal': signal_rewards_ave, 'route': route_rewards_ave,
                                 'sr_signal': sr_signal_rewards_ave, 'sr_route': sr_route_rewards_ave}
            return reward_norm_ave, total_rewards_ave
        rewards = []
        if self.problem == 'signal_route':
            rewards = rewards_signal + rewards_route
        elif self.problem == 'signal':
            rewards = rewards_signal
        elif self.problem == 'route':
            rewards = rewards_route
        return np.array(rewards)

    def _get_state(self, reset=False, problem=None):
        if reset:
            self._measure_state_step()
        if problem is not None:
            self.problem = problem
        self._clip_state()
        state_signal = []
        state_route = []
        if self.problem == 'signal' or self.problem == 'signal_route':
            for node_name in self.node_names:
                node = self.nodes[node_name]
                cur_state = [node.wave_state]
                for nnode_name in node.relevant_ss:
                    cur_state.append(self.nodes[nnode_name].wave_state)
                cur_state.append(node.wait_state)
                if self.problem == 'signal_route':
                    for rrc_name in node.relevant_sr:
                        cur_state.append(self.rcs[rrc_name].arrival_state)
                for nnode_name in node.relevant_ss:
                    fset = []
                    for f in self.nodes[nnode_name].fingerprint:
                        fset.append(f)
                    cur_state.append(fset)
                if self.problem == 'signal_route':
                    for rrc_name in node.relevant_sr:
                        fset = []
                        for f in self.rcs[rrc_name].fingerprint:
                            fset.append(f)
                        cur_state.append(fset)
                state_signal.append(np.concatenate(cur_state))
        if self.problem == 'route' or self.problem == 'signal_route':
            for rc_name in self.rc_names:
                rc = self.rcs[rc_name]
                cur_state = [rc.arrival_state]
                for rrc_name in rc.relevant_rr:
                    cur_state.append(self.rcs[rrc_name].arrival_state)
                temp = 0
                for edge in rc.down_edge:
                    rc.occupancy_state[temp] = rc.occupancy_state[temp] * rc.down_edge[edge]
                    temp += 1
                cur_state.append(rc.occupancy_state)
                if self.problem == 'signal_route':
                    for nnode_name in rc.relevant_rs:
                        cur_state.append(self.nodes[nnode_name].wave_state)
                for rrc_name in rc.relevant_rr:
                    fset = []
                    for f in self.rcs[rrc_name].fingerprint:
                        fset.append(f)
                    cur_state.append(fset)
                if self.problem == 'signal_route':
                    for nnode_name in rc.relevant_rs:
                        fset = []
                        for f in self.nodes[nnode_name].fingerprint:
                            fset.append(f)
                        cur_state.append(fset)
                state_route.append(np.concatenate(cur_state))
        state = []
        if self.problem == 'signal_route':
            state = state_signal + state_route
        elif self.problem == 'signal':
            state = state_signal
        elif self.problem == 'route':
            state = state_route
        return state

    def _init_agents(self):
        lane_to_detectors = {}
        if self.name != 'Sioux':
            for det_id in self.sim.lanearea.getIDList():
                lane_id = self.sim.lanearea.getLaneID(det_id)
                lane_to_detectors.setdefault(lane_id, []).append(det_id)

        nodes = {}
        for node_name in self._get_node_names():
            if node_name in self.relevant_ss_map:
                relevant_ss = self.relevant_ss_map[node_name]
            else:
                logging.info('node %s can not be found!' % node_name)
                relevant_ss = []
            if node_name in self.relevant_sr_map:
                relevant_sr = self.relevant_sr_map[node_name]
            else:
                logging.info('node %s can not be found!' % node_name)
                relevant_sr = []
            nodes[node_name] = Node(node_name, relevant_ss=relevant_ss, relevant_sr=relevant_sr, control=True)
            lanes_in = self.sim.trafficlight.getControlledLanes(node_name)
            nodes[node_name].lanes_in = lanes_in
            ilds_in = []
            for lane_name in lanes_in:
                if self.name == 'Sioux':
                    ild_name = lane_name
                    if ild_name not in ilds_in:
                        ilds_in.append(ild_name)
                else:
                    for det_id in lane_to_detectors.get(lane_name, []):
                        if det_id not in ilds_in:
                            ilds_in.append(det_id)
            nodes[node_name].ilds_in = ilds_in

        self.nodes = nodes
        self.node_names = sorted(list(nodes.keys()))
        self.rc_names = self._get_rc_names()
        rcs = {}
        for rc_name in self.rc_names:
            if rc_name in self.relevant_rr_map:
                relevant_rr = self.relevant_rr_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                relevant_rr = []
            if rc_name in self.relevant_rs_map:
                relevant_rs = self.relevant_rs_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                relevant_rs = []
            if rc_name in self.up_edge_map:
                up_edge = self.up_edge_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                up_edge = None
            if rc_name in self.down_edge_map:
                down_edge = self.down_edge_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                down_edge = {}
            if rc_name in self.forward_route_map:
                forward_route = self.forward_route_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                forward_route = []
            if rc_name in self.det_map:
                det = self.det_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                det = []
            if rc_name in self.f_id_map:
                f_id = self.f_id_map[rc_name]
            else:
                logging.info('rc %s can not be found!' % rc_name)
                f_id = []
            rcs[rc_name] = Rc(rc_name, f_id=f_id, relevant_rr=relevant_rr, relevant_rs=relevant_rs,
                              up_edge=up_edge, down_edge=down_edge, forward_route=forward_route,
                              det=det, control=True)
        self.rcs = rcs
        self.agent_type = []
        if self.problem == 'signal_route':
            for _ in range(len(self.node_names)):
                self.agent_type.append('signal')
            for _ in range(len(self.rc_names)):
                self.agent_type.append('route')
        elif self.problem == 'signal':
            for _ in range(len(self.node_names)):
                self.agent_type.append('signal')
        elif self.problem == 'route':
            for _ in range(len(self.rc_names)):
                self.agent_type.append('route')
        self._init_action_space()
        self._init_state_space()

    def _init_action_space(self):
        n_a_ls_signal = []
        for node_name in self.node_names:
            node = self.nodes[node_name]
            phase_id = self._get_node_phase_id(node_name)
            node.phase_id = phase_id
            node.n_a = self.phase_map.get_phase_num(phase_id)
            n_a_ls_signal.append(node.n_a)
        n_a_ls_route = []
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            route_num = self._get_rc_route_num(rc_name)
            rc.route_num = route_num
            rc.n_a = route_num
            n_a_ls_route.append(rc.n_a)
        self.n_a_ls = []
        if self.problem == 'signal_route':
            self.n_a_ls = n_a_ls_signal + n_a_ls_route
        elif self.problem == 'signal':
            self.n_a_ls = n_a_ls_signal
        elif self.problem == 'route':
            self.n_a_ls = n_a_ls_route
        self.n_a = int(np.prod(np.array(self.n_a_ls, dtype=object)))

    def _init_map(self):
        self.phase_map = None
        self.route_map = None
        self.relevant_ss_map = None
        self.relevant_rs_map = None
        self.relevant_sr_map = None
        self.relevant_rr_map = None
        self.up_edge_map = None
        self.down_edge_map = None
        self.forward_route_map = None
        self.f_id_map = None
        self.det_map = None
        self.state_names_signal = None
        self.state_names_route = None
        raise NotImplementedError()

    def _init_policy(self):
        policy_signal = []
        policy_route = []
        for node_name in self.node_names:
            phase_num = self.nodes[node_name].n_a
            p = 1. / phase_num
            policy_signal.append(np.array([p] * phase_num))
        for rc_name in self.rc_names:
            route_num = self.rcs[rc_name].n_a
            p = 1. / route_num
            policy_route.append(np.array([p] * route_num))
        if self.problem == 'signal_route':
            policy = policy_signal + policy_route
        elif self.problem == 'signal':
            policy = policy_signal
        elif self.problem == 'route':
            policy = policy_route
        return policy

    def _init_sim(self, seed, gui=False, load=False):
        sumocfg_file = self._init_sim_config(seed, load)
        if gui:
            app = 'sumo-gui'
        else:
            app = 'sumo'
        command = [checkBinary(app), '-c', sumocfg_file]
        command += ['--seed', str(seed)]
        command += ['--no-step-log', 'True']
        if self.name != 'real_net':
            command += ['--time-to-teleport', '600']
        else:
            command += ['--time-to-teleport', '300']
        command += ['--no-warnings', 'True']
        command += ['--duration-log.disable', 'True']
        command += ['--tripinfo-output',
                    self.output_path + ('%s_%s_trip.xml' % (self.name, self.problem))]
        command += ['--ignore-route-errors', 'true'] 
        command += ['--error-log', 'sumo_errors.txt']
        traci.start(command, port=self.port, label=str(self.port))
        self.sim = traci.getConnection(label=str(self.port))
        

    def _init_sim_config(self, seed=None, load=False):
        raise NotImplementedError()

    def _init_sim_traffic(self):
        return

    def _init_state_space(self):
        self._reset_state()
        self.n_s_ls = []
        self.n_w_ls = []
        self.n_f_ls = []
        n_s_ls_signal = []
        n_w_ls_signal = []
        n_f_ls_signal = []
        n_o_ls_signal = []
        for node_name in self.node_names:
            node = self.nodes[node_name]
            num_wave = node.num_state
            num_wait = node.num_state
            num_fingerprint = 0
            num_arrival = 0
            for nnode_name in node.relevant_ss:
                num_wave += self.nodes[nnode_name].num_state
                num_fingerprint += self.nodes[nnode_name].num_fingerprint
            if self.problem == 'signal_route':
                for rrc_name in node.relevant_sr:
                    num_arrival += self.rcs[rrc_name].num_state_arrival
                    num_fingerprint += self.rcs[rrc_name].num_fingerprint
            n_s_ls_signal.append(num_wave)
            n_w_ls_signal.append(num_wait)
            n_o_ls_signal.append(num_arrival)
            n_f_ls_signal.append(num_fingerprint)
        n_s_ls_route = []
        n_w_ls_route = []
        n_o_ls_route = []
        n_f_ls_route = []
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            num_arrival = rc.num_state_arrival
            num_occupancy = rc.num_state_occupancy
            num_fingerprint = 0
            num_wave = 0
            for rrc_name in rc.relevant_rr:
                num_arrival += self.rcs[rrc_name].num_state_arrival
                num_fingerprint += self.rcs[rrc_name].num_fingerprint
            if self.problem == 'signal_route':
                for nnode_name in rc.relevant_rs:
                    num_wave += self.nodes[nnode_name].num_state
                    num_fingerprint += self.nodes[nnode_name].num_fingerprint
            n_s_ls_route.append(num_arrival)
            n_w_ls_route.append(num_occupancy)
            n_o_ls_route.append(num_wave)
            n_f_ls_route.append(num_fingerprint)
        if self.problem == 'signal':
            self.n_s_ls = n_s_ls_signal
            self.n_w_ls = n_w_ls_signal
            self.n_f_ls = n_f_ls_signal
            self.n_s = np.sum(np.array(self.n_s_ls + self.n_w_ls + self.n_f_ls))
        elif self.problem == 'route':
            self.n_s_ls = n_s_ls_route
            self.n_w_ls = n_w_ls_route
            self.n_f_ls = n_f_ls_route
            self.n_s = np.sum(np.array(self.n_s_ls + self.n_w_ls + self.n_f_ls))
        elif self.problem == 'signal_route':
            self.n_s_ls_signal = n_s_ls_signal
            self.n_s_ls_route = n_s_ls_route
            self.n_w_ls_signal = n_w_ls_signal
            self.n_w_ls_route = n_w_ls_route
            self.n_o_ls_signal = n_o_ls_signal
            self.n_o_ls_route = n_o_ls_route
            self.n_f_ls_signal = n_f_ls_signal
            self.n_f_ls_route = n_f_ls_route
            self.n_s_ls = n_s_ls_signal + n_s_ls_route
            self.n_w_ls = n_w_ls_signal + n_w_ls_route
            self.n_o_ls = n_o_ls_signal + n_o_ls_route
            self.n_f_ls = n_f_ls_signal + n_f_ls_route
            self.n_s = np.sum(np.array(self.n_s_ls + self.n_w_ls + self.n_o_ls + self.n_f_ls))

    def _measure_reward_step(self):
        # For Sioux, ilds_in entries are lane IDs (legacy). For other
        # scenarios (e.g. Malta) ilds_in holds laneAreaDetector IDs and the
        # actual lane IDs are kept on node.lanes_in. The lane.* API only
        # accepts lane IDs, so pick the right list per scenario.
        use_lanes = self.name != 'Sioux'
        for node_name in self.node_names:
            node = self.nodes[node_name]
            waits = []
            vehwaits = []
            iter_ids = node.lanes_in if use_lanes else node.ilds_in
            for ild in iter_ids:
                halting = self.sim.lane.getLastStepHaltingNumber(ild)
                waits.append(min(40, halting) * self.control_interval_sec)
                max_pos = 0
                veh_wait = 0
                if halting == 0:
                    veh_wait = 0
                else:
                    vehs = self.sim.lane.getLastStepVehicleIDs(ild)
                    for vid in vehs:
                        pos = self.sim.vehicle.getLanePosition(vid)
                        if pos > max_pos:
                            max_pos = pos
                            veh_wait = self.sim.vehicle.getWaitingTime(vid)
                vehwaits.append(veh_wait)
            wait = np.sum(np.array(waits)) if len(waits) else 0
            vehwait = np.sum(np.array(vehwaits)) if len(vehwaits) else 0
            node.wait = wait
            node.vehwait = vehwait
            node.reward = -wait - self.coef_vehwait * vehwait
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            vehicles = []
            for det in rc.det:
                # Sioux scenario writes inductionloop IDs; Malta route
                # det_map stores `<edge>_0` style lane IDs (no inductionloops
                # exist in our additional file). Pick the right API.
                try:
                    if use_lanes:
                        vehicles += list(self.sim.lane.getLastStepVehicleIDs(det))
                    else:
                        vehicles += list(self.sim.inductionloop.getLastStepVehicleIDs(det))
                except Exception:
                    # Unknown id — skip silently rather than crash the episode.
                    continue
            arrived = 0
            for vehicle in vehicles:
                id = vehicle.split('_')
                if len(id) > 1 and id[1] == rc.f_id:
                    arrived += 1
            rc.arrived = arrived
            rc.reward = arrived

    def get_parameter_state(self):
        wave = []
        wait = []
        occupancy = []
        arrival = []
        for node_name in self.node_names:
            node = self.nodes[node_name]
            wave.append(node.wave_state)
            wait.append(node.wait_state)
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            arrival.append(rc.arrival_state)
            occupancy.append(rc.occupancy_state)
        wave_ave = np.mean([np.mean(w) for w in wave]) if wave else 0
        wait_ave = np.mean([np.mean(w) for w in wait]) if wait else 0
        arrival_ave = np.mean([np.mean(a) for a in arrival]) if arrival else 0
        occupancy_ave = np.mean([np.mean(o) for o in occupancy]) if occupancy else 0
        return {'wave': wave_ave, 'wait': wait_ave, 'arrival': arrival_ave, 'occupancy': occupancy_ave}

    def _clip_state(self):
        for node_name in self.node_names:
            node = self.nodes[node_name]
            node.wave_state = self._norm_clip_state(np.array(node.wave_state), self.norms['wave'], self.clips['wave'])
            node.wait_state = self._norm_clip_state(np.array(node.wait_state), self.norms['wait'], self.clips['wait'])
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            rc.arrival_state = self._norm_clip_state(np.array(rc.arrival_state), self.norms['arrival'], self.clips['arrival'])
            rc.occupancy_state = self._norm_clip_state(np.array(rc.occupancy_state), self.norms['occupancy'], self.clips['occupancy'])

    def _measure_state_step(self):
        lanearea_ids = set(self.sim.lanearea.getIDList()) if self.name != 'Sioux' else set()
        for node_name in self.node_names:
            node = self.nodes[node_name]
            for state_name in self.state_names_signal:
                if state_name == 'wave':
                    cur_state = []
                    for ild in node.ilds_in:
                        if self.name == 'Sioux':
                            cur_wave = self.sim.lane.getLastStepVehicleNumber(ild)
                        else:
                            if ild in lanearea_ids:
                                cur_wave = self.sim.lanearea.getLastStepVehicleNumber(ild)
                            else:
                                logging.warning('Missing detector: %s' % ild)
                                cur_wave = 0
                        cur_state.append(cur_wave)
                    cur_state = np.array(cur_state)
                else:
                    cur_state = []
                    for ild in node.ilds_in:
                        if self.name == 'Sioux':
                            cur_wait = self.sim.lane.getLastStepHaltingNumber(ild) * self.control_interval_sec
                        else:
                            if ild in lanearea_ids:
                                cur_wait = self.sim.lanearea.getLastStepHaltingNumber(ild) * self.control_interval_sec
                            else:
                                logging.warning('Missing detector: %s' % ild)
                                cur_wait = 0
                        cur_state.append(cur_wait)
                    cur_state = np.array(cur_state)
                if state_name == 'wave':
                    node.wave_state = cur_state
                else:
                    node.wait_state = cur_state
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            for state_name in self.state_names_route:
                if state_name == 'arrival':
                    cur_arrival = 0
                    Vehicle_Set = self.sim.edge.getLastStepVehicleIDs(rc.up_edge)
                    for vehicle in Vehicle_Set:
                        id = vehicle.split('_')
                        if id[1] == rc.f_id:
                            cur_arrival += 1
                    cur_state = np.array([cur_arrival])
                elif state_name == 'occupancy':
                    cur_state = []
                    for edge in rc.down_edge:
                        cur_occupancy = self.sim.edge.getLastStepVehicleNumber(edge)
                        cur_state.append(cur_occupancy)
                    cur_state = np.array(cur_state)
                if state_name == 'arrival':
                    rc.arrival_state = cur_state
                elif state_name == 'occupancy':
                    rc.occupancy_state = cur_state

    def _measure_traffic_step(self):
        cars = self.sim.vehicle.getIDList()
        num_tot_car = len(cars)
        num_in_car = self.sim.simulation.getDepartedNumber()
        num_out_car = self.sim.simulation.getArrivedNumber()
        if num_tot_car > 0:
            avg_waiting_time = np.mean([self.sim.vehicle.getWaitingTime(car) for car in cars])
            avg_speed = np.mean([self.sim.vehicle.getSpeed(car) for car in cars])
        else:
            avg_speed = 0
            avg_waiting_time = 0
        queues = []
        for node_name in self.node_names:
            for ild in self.nodes[node_name].ilds_in:
                try:
                    queues.append(self.sim.lane.getLastStepHaltingNumber(ild))
                except Exception:
                    pass
        avg_queue = np.mean(np.array(queues)) if queues else 0.0
        std_queue = np.std(np.array(queues))  if queues else 0.0
        cur_traffic = {'episode': self.cur_episode,
                       'time_sec': self.cur_sec,
                       'number_total_car': num_tot_car,
                       'number_departed_car': num_in_car,
                       'number_arrived_car': num_out_car,
                       'avg_wait_sec': avg_waiting_time,
                       'avg_speed_mps': avg_speed,
                       'std_queue': std_queue,
                       'avg_queue': avg_queue}
        self.traffic_data.append(cur_traffic)

    @staticmethod
    def _norm_clip_state(x, norm, clip=-1):
        x = x / norm
        return x if clip < 0 else np.clip(x, 0, clip)

    @staticmethod
    def _norm_clip_reward(x, norm, clip=-1):
        x = x / norm
        return x if clip < 0 else np.clip(x, -clip, clip)

    def _reset_state(self):
        for node_name in self.node_names:
            node = self.nodes[node_name]
            node.prev_action = 0
            node.num_fingerprint = node.n_a - 1
            node.num_state = self._get_node_state_num(node)
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            rc.prev_action = 0
            rc.num_fingerprint = rc.n_a - 1
            rc.num_state_arrival = self._get_rc_state_num()
            rc.num_state_occupancy = len(rc.down_edge)

    def _set_phase(self, action):
        for i in range(len(self.node_names)):
            node_name = self.node_names[i]
            a = list(action)[i]
            phase = self._get_node_phase(a, node_name)
            self.sim.trafficlight.setRedYellowGreenState(node_name, phase)
            self.sim.trafficlight.setPhaseDuration(node_name, self.control_interval_sec)

    def _set_route(self, action):
        # Lazy import so we don't add a hard dep at module load time.
        try:
            from traci.exceptions import TraCIException
        except Exception:
            TraCIException = Exception
        for i in range(len(self.rc_names)):
            rc_name = self.rc_names[i]
            rc = self.rcs[rc_name]
            a = list(action)[i]
            Vehicle_Set = self.sim.edge.getLastStepVehicleIDs(rc.up_edge)
            vehicles = []
            for vehicle in Vehicle_Set:
                id = vehicle.split('_')
                if id[1] == rc.f_id:
                    vehicles.append(vehicle)
            route = rc.forward_route
            pi = [self.CAV_rate, 1 - self.CAV_rate]
            for i in range(len(vehicles)):
                p = np.random.choice(np.arange(len(pi)), p=pi)
                if p == 0:
                    target_route = route[0] if a == 0 else route[1]
                    # SUMO refuses a route swap if the vehicle's current edge
                    # is not contained in the new route. With auto-fixed
                    # corridors whose two alternatives don't share a common
                    # starting edge (or bidirectional pairs going opposite
                    # ways), this is common. Skip the swap silently — the
                    # vehicle keeps its original route, which is safe.
                    try:
                        self.sim.vehicle.setRoute(vehicles[i], target_route)
                    except TraCIException:
                        continue

    def _simulate(self):
        import traci.constants as tc
        self.cur_sec += self.control_interval_sec
        self.sim.simulation.step(time=float(self.cur_sec))
        arrived_number = self.sim.simulation.getArrivedNumber()
        departed_number = self.sim.simulation.getDepartedNumber()
        # Subscribe each newly departed vehicle to speed (one-time per vehicle).
        # SUMO removes departed vehicles from results automatically.
        for vid in self.sim.simulation.getDepartedIDList():
            self.sim.vehicle.subscribe(vid, [tc.VAR_SPEED])
        # One bulk call replaces thousands of per-edge TraCI round-trips.
        veh_results = self.sim.vehicle.getAllSubscriptionResults()
        halting = sum(1 for v in veh_results.values() if v.get(tc.VAR_SPEED, 1.0) < 0.1)
        waiting_time = halting * self.control_interval_sec
        travel_time  = len(veh_results) * self.control_interval_sec
        self._measure_state_step()
        self._measure_reward_step()
        if self.is_record:
            self._measure_traffic_step()
        return arrived_number, departed_number, waiting_time, travel_time

    def collect_tripinfo(self):
        trip_file = self.output_path + ('%s_%s_trip.xml' % (self.name, self.problem))
        print(trip_file)
        tree = ET.ElementTree(file=trip_file)
        for child in tree.getroot():
            cur_trip = child.attrib
            cur_dict = {}
            cur_dict['episode'] = self.cur_episode
            cur_dict['id'] = cur_trip['id']
            cur_dict['depart_sec'] = cur_trip['depart']
            cur_dict['arrival_sec'] = cur_trip['arrival']
            cur_dict['duration_sec'] = cur_trip['duration']
            cur_dict['wait_step'] = cur_trip['waitingCount']
            cur_dict['wait_sec'] = cur_trip['waitingTime']
            self.trip_data.append(cur_dict)
        os.remove(trip_file)

    def init_data(self, is_record, output_path, diff=None):
        self.is_record = is_record
        if diff is None:
            self.output_path = output_path
        else:
            self.output_path = 'Sioux\\%s\\eva_data\\%.1f\\' % (self.problem, diff)
        if self.is_record:
            self.traffic_data = []
            self.control_data = []
            self.trip_data = []

    def init_test_seeds(self, test_seeds):
        self.test_num = len(test_seeds)
        self.test_seeds = test_seeds

    def output_data(self):
        if not self.is_record:
            logging.error('Env: no record to output!')
            return
        pd.DataFrame(self.control_data).to_csv(self.output_path + ('%s_%s_control.csv' % (self.name, self.problem)))
        pd.DataFrame(self.traffic_data).to_csv(self.output_path + ('%s_%s_traffic.csv' % (self.name, self.problem)))
        pd.DataFrame(self.trip_data).to_csv(self.output_path + ('%s_%s_trip.csv' % (self.name, self.problem)))

    def reset(self, gui=False, test_ind=0, load=False, separate=False):
        self._reset_state()
        if self.train_mode:
            seed = self.seed
        else:
            seed = self.test_seeds[test_ind]
        if load == False:
            self._init_sim(seed, gui=gui)
        else:
            self._init_sim(seed, gui=gui, load=load)
        self.cur_sec = 0
        self.cur_episode += 1
        if self.agent == 'ma2c':
            self.update_fingerprint(self._init_policy())
        self._init_sim_traffic()
        self.seed += 1
        if separate == True:
            state = self._get_state(reset=True, problem='signal') + self._get_state(reset=True, problem='route')
            self.problem = 'signal_route'
            return state
        else:
            return self._get_state(reset=True)

    def terminate(self):
        self.sim.close()

    def get_parameter(self, scenario):
        state_data = []
        reward_data = []
        total_reward_data = []
        self._init_sim(self.seed)
        self.cur_sec = 0
        for i in range(int(self.episode_length_sec / self.control_interval_sec)):
            self._simulate()
            get_parameter = True
            state_ave = self.get_parameter_state()
            state_data.append(state_ave)
            reward_ave, total_rewards_ave = self._get_reward(get_parameter=get_parameter)
            reward_data.append(reward_ave)
            total_reward_data.append(total_rewards_ave)
        path = scenario + os.sep
        pd.DataFrame(state_data).to_csv(path + 'state.csv')
        pd.DataFrame(reward_data).to_csv(path + 'reward.csv')
        pd.DataFrame(total_reward_data).to_csv(path + 'total_reward.csv')

    def step_separate(self, action_signal, action_route):
        self._set_route(action_route)
        self._set_phase(action_signal)
        _, _, _, _ = self._simulate()
        signal_state = self._get_state(problem='signal')
        route_state = self._get_state(problem='route')
        self.problem = 'signal_route'
        state = signal_state + route_state
        done = False
        if self.cur_sec >= self.episode_length_sec:
            done = True
        return state, done

    def step(self, action):
        if self.problem == 'signal_route':
            self._set_route(action[len(self.node_names):])
            self._set_phase(action[:len(self.node_names)])
        elif self.problem == 'signal':
            self._set_phase(action)
        elif self.problem == 'route':
            self._set_route(action)
        arrived_number, departed_number, waiting_time, travel_time = self._simulate()
        state = self._get_state()
        reward = self._get_reward()
        done = False
        if self.cur_sec >= self.episode_length_sec:
            done = True
        global_reward = np.sum(reward)
        if self.is_record:
            action_r = ','.join(['%d' % a for a in action])
            cur_control = {'episode': self.cur_episode,
                           'time_sec': self.cur_sec,
                           'step': self.cur_sec / self.control_interval_sec,
                           'action': action_r,
                           'reward': global_reward}
            self.control_data.append(cur_control)
        agent_reward = reward
        reward = self._local_reward_step(reward)
        reward = self._clip_reward(reward)
        if not self.train_mode:
            return state, reward, done, global_reward, arrived_number, departed_number, waiting_time, travel_time, agent_reward
        return state, reward, done, global_reward, arrived_number, departed_number, waiting_time, travel_time, agent_reward

    def _clip_reward(self, reward):
        if self.problem != 'route':
            rewards_signal = reward[:len(self.node_names)]
            rewards_signal = self._norm_clip_reward(np.array(rewards_signal), self.norms['reward_signal'], self.clips['reward'])
            if self.problem == 'signal_route':
                rewards_route = reward[len(self.node_names):]
                rewards_route = self._norm_clip_reward(np.array(rewards_route), self.norms['reward_route'], self.clips['reward'])
        elif self.problem == 'route':
            rewards_route = reward[:len(self.rc_names)]
            rewards_route = self._norm_clip_reward(np.array(rewards_route), self.norms['reward_route'], self.clips['reward'])
        rewards = []
        if self.problem == 'signal_route':
            rewards = list(rewards_signal) + list(rewards_route)
        elif self.problem == 'signal':
            rewards = list(rewards_signal)
        elif self.problem == 'route':
            rewards = list(rewards_route)
        return np.array(rewards)

    def _local_reward_step(self, reward):
        new_reward = []
        if self.problem != 'route':
            for i in range(len(self.node_names)):
                node_name = self.node_names[i]
                cur_reward = reward[i]
                for nnode_name in self.nodes[node_name].relevant_ss:
                    j = self.node_names.index(nnode_name)
                    cur_reward += self.coop_gamma_ss * reward[j]
                if self.problem == 'signal_route':
                    for rrc_name in self.nodes[node_name].relevant_sr:
                        j = self.rc_names.index(rrc_name)
                        cur_reward += self.coop_gamma_sr * self.nodes[node_name].relevant_sr[rrc_name] * reward[len(self.node_names) + j]
                new_reward.append(cur_reward)
        if self.problem != 'signal':
            for i in range(len(self.rc_names)):
                rc_name = self.rc_names[i]
                if self.problem == 'signal_route':
                    cur_reward = reward[len(self.node_names) + i]
                    for rrc_name in self.rcs[rc_name].relevant_rr:
                        j = self.rc_names.index(rrc_name)
                        cur_reward += self.coop_gamma_rr * reward[len(self.node_names) + j]
                    for nnode_name in self.rcs[rc_name].relevant_rs:
                        j = self.node_names.index(nnode_name)
                        cur_reward += self.coop_gamma_rs * self.rcs[rc_name].relevant_rs[nnode_name] * reward[j]
                else:
                    cur_reward = reward[i]
                    for rrc_name in self.rcs[rc_name].relevant_rr:
                        j = self.rc_names.index(rrc_name)
                        cur_reward += self.coop_gamma_rr * reward[j]
                new_reward.append(cur_reward)
        return np.array(new_reward)

    def update_fingerprint(self, policy):
        if self.problem != 'route':
            for i in range(len(self.node_names)):
                node_name = self.node_names[i]
                pi = policy[i]
                self.nodes[node_name].fingerprint = np.array(pi)[:-1]
            if self.problem == 'signal_route':
                for i in range(len(self.rc_names)):
                    rc_name = self.rc_names[i]
                    pi = policy[len(self.node_names) + i]
                    self.rcs[rc_name].fingerprint = np.array(pi)[:-1]
        else:
            for i in range(len(self.rc_names)):
                rc_name = self.rc_names[i]
                pi = policy[i]
                self.rcs[rc_name].fingerprint = np.array(pi)[:-1]


## 7. Malta Environment (`envs/Malta_env.py`)

Concrete `TrafficSimulator` subclass for the Malta network.

* Loads network topology from a pickle cache (`Malta/data/malta_cache_v2.pkl`)
  so the expensive XML parse only happens once
* **`MaltaPhase`** — wraps the `PHASES` dict from `build_file`
* **`MaltaRoute`** — builds all corridor maps (up/down edges, detectors,
  fingerprint IDs) from the validated route list
* **`MaltaController`** — greedy baseline: picks the phase with the highest
  combined queue across all incoming lanes
* **`MaltaEnv`** — overrides `_init_map`, `_get_node_names`,
  `_get_node_phase_id`, `_init_action_space`, etc. to wire the Malta-specific
  structures into the base simulator


In [10]:
"""

envs/Malta_env.py

Malta environment adapted from Sioux_env.py

"""



import logging

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from envs.env import PhaseMap, PhaseSet, RouteMap, RouteSet, TrafficSimulator
from Malta.build_file import gen_rou_file, get_routes, relevant_ss, get_nodes, get_phases, get_flows

import pickle, os

_CACHE_FILE = os.path.join(os.getcwd(), 'Malta', 'data', 'malta_cache_v2.pkl')  # notebook-safe

def _load_or_build_cache():
    if os.path.exists(_CACHE_FILE):
        with open(_CACHE_FILE, 'rb') as f:
            return pickle.load(f)
    data = {
        'nodes':  get_nodes(),
        'routes': get_routes(),
        'phases': get_phases(),
        'flows':  get_flows(),
    }
    with open(_CACHE_FILE, 'wb') as f:
        pickle.dump(data, f)
    return data

_cache = _load_or_build_cache()
NODE, NOTLNODE = _cache['nodes']
routes0, rc_num = _cache['routes']
RC_NAMES = ['rc%d' % i for i in range(1, rc_num + 1)]
PHASES, PHASE_NODE_MAP = _cache['phases']






sns.set_color_codes()



STATE_NAMES_SIGNAL = ['wave', 'wait']

STATE_NAMES_ROUTE = ['arrival', 'occupancy']

STATE_ARRIVAL = 1

STATE_OCCUPANCY = 2



'''NODE, NOTLNODE = get_nodes()

routes0, rc_num = get_routes()



PHASES, PHASE_NODE_MAP = get_phases()
'''
RC_NAMES = ['rc%d' % i for i in range(1, rc_num + 1)]


rs_rate = 0.5

thres = 0.05





class MaltaPhase(PhaseMap):

    def __init__(self):

        self.phases = {}

        for key, val in PHASES.items():

            self.phases[key] = PhaseSet(val)





class MaltaRoute(RouteMap):

    def __init__(self):

        self.relevant_rr_map = {}

        self.relevant_rs_map = {}

        self.up_edge_map = {}

        self.down_edge_map = {}

        self.forward_route_map = {}

        self.f_id_map = {}

        self.det_map = {}

        self.routes = {}



        for i, route_set in enumerate(routes0):

            rc_id = 'rc%d' % (i + 1)



            if len(route_set) != 2:

                raise ValueError(

                    "MaltaRoute: expected 2 alternatives per corridor, "

                    "got %d for corridor %d" % (len(route_set), i)

                )



            route_a = route_set[0]

            route_b = route_set[1]



            self.routes[rc_id] = RouteSet(route_set)

            self.up_edge_map[rc_id] = route_a[0] if route_a else ''



            down = {}

            for j, eid in enumerate(route_a):

                down[eid] = rs_rate ** j

            for j, eid in enumerate(route_b):

                if eid not in down:

                    down[eid] = rs_rate ** j

                else:

                    down[eid] = max(down[eid], rs_rate ** j)

            self.down_edge_map[rc_id] = down



            self.forward_route_map[rc_id] = [route_a, route_b]



            self.det_map[rc_id] = [

                route_a[-1] + '_0',

                route_b[-1] + '_0'

            ]



            self.f_id_map[rc_id] = str(i + 1)



            self.relevant_rs_map[rc_id] = {}

            all_edges = route_a + route_b

            for j, eid in enumerate(all_edges):

                weight = rs_rate ** j

                if weight < thres:

                    break

                if eid not in self.relevant_rs_map[rc_id]:

                    self.relevant_rs_map[rc_id][eid] = weight

                else:

                    self.relevant_rs_map[rc_id][eid] = max(

                        self.relevant_rs_map[rc_id][eid], weight

                    )



            self.relevant_rr_map[rc_id] = []



        self.relevant_sr_map = {n: {} for n in NODE}





class MaltaController:

    def __init__(self, node_names, rc_names):

        self.name = 'greedy'

        self.node_names = node_names

        self.rc_names = rc_names



    def forward(self, obs):

        actions = []

        for ob, node_name in zip(obs, self.node_names):

            actions.append(self.greedy(ob, node_name))

        return actions



    def greedy(self, ob, node_name):

        if len(ob) < 6:

            return 0

        flows = [ob[0] + ob[3], ob[2] + ob[5], ob[1] + ob[4],

                 ob[1] + ob[2], ob[4] + ob[5]]

        return np.argmax(np.array(flows))





class MaltaEnv(TrafficSimulator):
    def init_data(self, is_record, output_path, diff=None):
        self.is_record = is_record
        self.output_path = output_path
        if self.is_record:
            self.traffic_data = []
            self.control_data = []
            self.trip_data = []

    def init_test_seeds(self, test_seeds):
        self.test_num = len(test_seeds)
        self.test_seeds = test_seeds

    def terminate(self):
        self.sim.close()

    def reset(self, gui=False, test_ind=0, load=False, separate=False):
        self._reset_state()
        if self.train_mode:
            seed = self.seed
        else:
            seed = self.test_seeds[test_ind]
        self._init_sim(seed, gui=gui)
        self.cur_sec = 0
        self.cur_episode += 1
        if self.agent == 'ma2c':
            self.update_fingerprint(self._init_policy())
        self._init_sim_traffic()
        self.seed += 1
        return self._get_state(reset=True)

    def output_data(self):
        if not self.is_record:
            return
        import pandas as pd
        pd.DataFrame(self.control_data).to_csv(self.output_path + (self.name + '_' + self.problem + '_control.csv'))
        pd.DataFrame(self.traffic_data).to_csv(self.output_path + (self.name + '_' + self.problem + '_traffic.csv'))
        pd.DataFrame(self.trip_data).to_csv(self.output_path + (self.name + '_' + self.problem + '_trip.csv'))

    def init_data(self, is_record, output_path, diff=None):
        self.is_record = is_record
        self.output_path = output_path
        if self.is_record:
            self.traffic_data = []
            self.control_data = []
            self.trip_data = []


    def __init__(self, config, problem, output_path='', is_record=False, port=0):
        self.demand_flows = _cache['flows']
        super().__init__(config, problem, output_path, is_record, port=port)


    def _get_node_names(self):

        """Return only TL nodes that exist in phase_node_map."""

        all_nodes = self.sim.trafficlight.getIDList()

        valid = [n for n in all_nodes if n in self.phase_node_map]

        skipped = [n for n in all_nodes if n not in self.phase_node_map]

        for n in skipped:

            logging.info('Skipping TL node not in phase_node_map: %s' % n)

        return valid



    def _get_node_phase_id(self, node_name):

        """Return phase_id for node, finding best lane-count match if needed."""

        if node_name in self.phase_node_map:

            return self.phase_node_map[node_name]

        fallback = next(iter(self.phase_node_map))

        logging.warning('node %s not in phase_node_map, using fallback %s'

                        % (node_name, fallback))

        return fallback



    def _find_matching_phase(self, lane_num):

        """Find a phase_id whose encoded lane count matches lane_num."""

        # exact match first

        for phase_id in self.phase_map.phases:

            if self.phase_map.get_lane_num(phase_id) == lane_num:

                return phase_id

        # closest match

        best_id = None

        best_diff = float('inf')

        for phase_id in self.phase_map.phases:

            diff = abs(self.phase_map.get_lane_num(phase_id) - lane_num)

            if diff < best_diff:

                best_diff = diff

                best_id = phase_id

        return best_id



    def _get_node_state_num(self, node):

        lane_num = len(node.lanes_in)

        current_phase_id = node.phase_id



        if self.phase_map.get_lane_num(current_phase_id) != lane_num:

            best_phase = self._find_matching_phase(lane_num)

            if best_phase:

                logging.info(

                    'node %s: lane count %d does not match phase %s (%d lanes), switching to phase %s'

                    % (node.name, lane_num, current_phase_id,

                    self.phase_map.get_lane_num(current_phase_id), best_phase)

                )

                node.phase_id = best_phase



        return len(node.ilds_in)



    def _get_rc_route_num(self, rc_name):

        return 2



    def _get_rc_names(self):

        return RC_NAMES



    def _get_rc_state_num(self):
        return STATE_ARRIVAL

    # ADD THIS RIGHT HERE ↓
    def _init_action_space(self):
        n_a_ls_signal = []
        for node_name in self.node_names:
            node = self.nodes[node_name]
            phase_id = self._get_node_phase_id(node_name)
            node.phase_id = phase_id
            node.n_a = self.phase_map.get_phase_num(phase_id)
            n_a_ls_signal.append(node.n_a)

        n_a_ls_route = []
        for rc_name in self.rc_names:
            rc = self.rcs[rc_name]
            rc.route_num = self._get_rc_route_num(rc_name)
            rc.n_a = rc.route_num
            n_a_ls_route.append(rc.n_a)

        if self.problem == 'signal_route':
            self.n_a_ls = n_a_ls_signal + n_a_ls_route
        elif self.problem == 'signal':
            self.n_a_ls = n_a_ls_signal
        elif self.problem == 'route':
            self.n_a_ls = n_a_ls_route

        # Use max instead of np.prod to avoid combinatorial explosion (e.g. 3^190)
        self.n_a = max(self.n_a_ls)

    



    def _init_relevant_ss_map(self):

        return relevant_ss()



    def _init_map(self):

        self.phase_node_map = PHASE_NODE_MAP

        self.phase_map = MaltaPhase()

        self.route_map = MaltaRoute()

        self.relevant_ss_map = self._init_relevant_ss_map()

        self.relevant_rs_map = self.route_map.relevant_rs_map

        self.relevant_sr_map = self.route_map.relevant_sr_map

        self.relevant_rr_map = self.route_map.relevant_rr_map

        self.up_edge_map = self.route_map.up_edge_map

        self.down_edge_map = self.route_map.down_edge_map

        self.forward_route_map = self.route_map.forward_route_map

        self.f_id_map = self.route_map.f_id_map

        self.det_map = self.route_map.det_map

        self.state_names_signal = STATE_NAMES_SIGNAL

        self.state_names_route = STATE_NAMES_ROUTE

        # clean relevant_rs_map to only keep TL node keys

        tl_set = set(NODE)

        for rc in self.relevant_rs_map:

            self.relevant_rs_map[rc] = {

                k: v for k, v in self.relevant_rs_map[rc].items()

                if k in tl_set

            }



    def _init_sim_config(self, seed, load):

        return gen_rou_file(

            self.data_path,

            self.demand_flows,

            seed=seed,

            thread=self.sim_thread,

            load=load

        )



    def plot_stat(self, rewards):

        self.state_stat['reward'] = rewards

        for name, data in self.state_stat.items():

            fig = plt.figure(figsize=(8, 6))

            plot_cdf(data)

            plt.ylabel(name)

            fig.savefig(self.output_path + self.name + '_' + name + '.png')





def plot_cdf(X, c='b', label=None):

    sorted_data = np.sort(X)

    yvals = np.arange(len(sorted_data)) / float(len(sorted_data) - 1)

    plt.plot(sorted_data, yvals, color=c, label=label)

## 8. Agent Neural-Network Utilities (`agents/utils.py`)

Low-level TF1 building blocks shared by all policy classes.

| Symbol | Type | Purpose |
|--------|------|---------|
| `ortho_init` | initialiser | Orthogonal weight init (stable A2C training) |
| `norm_init` | initialiser | Fan-in normalised init |
| `fc(x, scope, n_out)` | layer | Fully-connected + ReLU |
| `conv(x, scope, ...)` | layer | 1-D or 2-D convolution + ReLU |
| `lstm(xs, dones, s, scope)` | layer | Custom LSTM that resets state on episode end (`done=1`) |
| `OnPolicyBuffer` | class | Stores (obs, act, reward, value, done) tuples; computes returns + advantages |
| `ReplayBuffer` | class | Fixed-size circular off-policy buffer (used by DQN variants) |
| `Scheduler` | class | Linear or constant learning-rate schedule |


In [11]:
import numpy as np
import random
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

DEFAULT_SCALE = np.sqrt(2)
DEFAULT_MODE = 'fan_in'

def ortho_init(scale=DEFAULT_SCALE, mode=None):
    def _ortho_init(shape, dtype, partition_info=None):
        # lasagne ortho init for tf
        shape = tuple(shape)
        if len(shape) == 2: # fc: in, out
            flat_shape = shape
        elif (len(shape) == 3) or (len(shape) == 4): # 1d/2dcnn: (in_h), in_w, in_c, out
            flat_shape = (np.prod(shape[:-1]), shape[-1])
        a = np.random.standard_normal(flat_shape)
        u, _, v = np.linalg.svd(a, full_matrices=False)
        q = u if u.shape == flat_shape else v # pick the one with the correct shape
        q = q.reshape(shape)
        return (scale * q).astype(np.float32)
    return _ortho_init


def norm_init(scale=DEFAULT_SCALE, mode=DEFAULT_MODE):
    def _norm_init(shape, dtype, partition_info=None):
        shape = tuple(shape)
        if len(shape) == 2:
            n_in = shape[0]
        elif (len(shape) == 3) or (len(shape) == 4):
            n_in = np.prod(shape[:-1])
        a = np.random.standard_normal(shape)
        if mode == 'fan_in':
            n = n_in
        elif mode == 'fan_out':
            n = shape[-1]
        elif mode == 'fan_avg':
            n = 0.5 * (n_in + shape[-1])
        return (scale * a / np.sqrt(n)).astype(np.float32)

DEFAULT_METHOD = ortho_init
"""
layers
"""
def conv(x, scope, n_out, f_size, stride=1, pad='VALID', f_size_w=None, act=tf.nn.relu,
         conv_dim=1, init_scale=DEFAULT_SCALE, init_mode=None, init_method=DEFAULT_METHOD):
    with tf.variable_scope(scope):
        b = tf.get_variable("b", [n_out], initializer=tf.constant_initializer(0.0))
        if conv_dim == 1:
            n_c = x.shape[2].value
            w = tf.get_variable("w", [f_size, n_c, n_out],
                                initializer=init_method(init_scale, init_mode))
            z = tf.nn.conv1d(x, w, stride=stride, padding=pad) + b
        elif conv_dim == 2:
            n_c = x.shape[3].value
            if f_size_w is None:
                f_size_w = f_size
            w = tf.get_variable("w", [f_size, f_size_w, n_c, n_out],
                                initializer=init_method(init_scale, init_mode))
            z = tf.nn.conv2d(x, w, strides=[1, stride, stride, 1], padding=pad) + b
        return act(z)


def fc(x, scope, n_out, act=tf.nn.relu, init_scale=DEFAULT_SCALE,
       init_mode=DEFAULT_MODE, init_method=DEFAULT_METHOD):
    with tf.variable_scope(scope):
        n_in = x.shape[1].value
        w = tf.get_variable("w", [n_in, n_out],
                            initializer=init_method(init_scale, init_mode))
        b = tf.get_variable("b", [n_out], initializer=tf.constant_initializer(0.0))
        z = tf.matmul(x, w) + b
        return act(z)


def batch_to_seq(x):
    n_step = x.shape[0].value
    if len(x.shape) == 1:
        x = tf.expand_dims(x, -1)
    return tf.split(axis=0, num_or_size_splits=n_step, value=x)


def seq_to_batch(x):
    return tf.concat(axis=0, values=x)


def lstm(xs, dones, s, scope, init_scale=DEFAULT_SCALE, init_mode=DEFAULT_MODE,
         init_method=DEFAULT_METHOD):
    xs = batch_to_seq(xs)
    # need dones to reset states
    dones = batch_to_seq(dones)
    n_in = xs[0].shape[1].value
    n_out = s.shape[0] // 2
    with tf.variable_scope(scope):
        wx = tf.get_variable("wx", [n_in, n_out*4],
                             initializer=init_method(init_scale, init_mode))
        wh = tf.get_variable("wh", [n_out, n_out*4],
                             initializer=init_method(init_scale, init_mode))
        b = tf.get_variable("b", [n_out*4], initializer=tf.constant_initializer(0.0))
    s = tf.expand_dims(s, 0)
    c, h = tf.split(axis=1, num_or_size_splits=2, value=s)
    for ind, (x, done) in enumerate(zip(xs, dones)):
        c = c * (1-done)
        h = h * (1-done)
        z = tf.matmul(x, wx) + tf.matmul(h, wh) + b
        i, f, o, u = tf.split(axis=1, num_or_size_splits=4, value=z)
        i = tf.nn.sigmoid(i)
        f = tf.nn.sigmoid(f)
        o = tf.nn.sigmoid(o)
        u = tf.tanh(u)
        c = f*c + i*u
        h = o*tf.tanh(c)
        xs[ind] = h
    s = tf.concat(axis=1, values=[c, h])
    return seq_to_batch(xs), tf.squeeze(s)


def test_layers():
    print(tf.__version__)
    tf.reset_default_graph()
    sess = tf.Session()
    n_step = 5
    fc_x = tf.placeholder(tf.float32, [None, 10])
    lstm_x = tf.placeholder(tf.float32, [n_step, 2])
    lstm_done = tf.placeholder(tf.float32, [n_step])
    lstm_s = tf.placeholder(tf.float32, [20])
    conv1_x = tf.placeholder(tf.float32, [None, 8, 1])
    conv2_x = tf.placeholder(tf.float32, [None, 8, 8, 1])
    fc_out = fc(fc_x, 'fc', 10)
    lstm_out, lstm_ns = lstm(lstm_x, lstm_done, lstm_s, 'lstm')
    conv1_out = conv(conv1_x, 'conv1', 10, 4, conv_dim=1)
    conv2_out = conv(conv2_x, 'conv2', 10, 4, conv_dim=2)
    sess.run(tf.global_variables_initializer())
    inputs = {'fc': {fc_x:np.random.randn(n_step, 10)},
              'lstm_done': {lstm_x:np.zeros((n_step, 2)),
                            lstm_done:np.ones(n_step),
                            lstm_s:np.random.randn(20)},
              'lstm': {lstm_x:np.random.randn(n_step, 2),
                       lstm_done:np.zeros(n_step),
                       lstm_s:np.random.randn(20)},
              'conv1': {conv1_x:np.random.randn(n_step, 8, 1)},
              'conv2': {conv2_x:np.random.randn(n_step, 8, 8, 1)}}
    outputs = {'fc': [fc_out], 'lstm_done': [lstm_out, lstm_ns],
               'conv1': [conv1_out], 'conv2': [conv2_out],
               'lstm': [lstm_out, lstm_ns]}
    for scope in ['fc', 'lstm', 'conv1', 'conv2']:
        print(scope)
        wts = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope)
        for wt in wts:
            wt_val = wt.eval(sess)
            print(wt_val.shape)
            print(np.mean(wt_val), np.std(wt_val), np.min(wt_val), np.max(wt_val))
    print('=====================================')
    for x_name in inputs:
        print(x_name)
        out = sess.run(outputs[x_name], inputs[x_name])
        if x_name.startswith('lstm'):
            print(out[0])
            print(out[1])
        else:
            print(out[0].shape)

"""
buffers
"""
class TransBuffer:
    def reset(self):
        self.buffer = []

    @property
    def size(self):
        return len(self.buffer)

    def add_transition(self, ob, a, r, *_args, **_kwargs):
        raise NotImplementedError()

    def sample_transition(self, *_args, **_kwargs):
        raise NotImplementedError()


class OnPolicyBuffer(TransBuffer):
    def __init__(self, gamma):
        self.gamma = gamma
        self.reset()

    def reset(self, done=False):
        # the done before each step is required
        self.obs = []
        self.acts = []
        self.rs = []
        self.vs = []
        self.dones = [done]

    def add_transition(self, ob, a, r, v, done):
        self.obs.append(ob)
        self.acts.append(a)
        self.rs.append(r)
        self.vs.append(v)
        self.dones.append(done)

    def _add_R_Adv(self, R):
        Rs = []
        Advs = []
        # use post-step dones here
        for r, v, done in zip(self.rs[::-1], self.vs[::-1], self.dones[:0:-1]):
            R = r + self.gamma * R * (1.-done)
            Adv = R - v
            Rs.append(R)
            Advs.append(Adv)
        Rs.reverse()
        Advs.reverse()
        self.Rs = Rs
        self.Advs = Advs

    def sample_transition(self, R, discrete=True):
        self._add_R_Adv(R)
        obs = np.array(self.obs, dtype=np.float32)
        if discrete:
            acts = np.array(self.acts, dtype=np.int32)
        else:
            acts = np.array(self.acts, dtype=np.float32)
        Rs = np.array(self.Rs, dtype=np.float32)
        Advs = np.array(self.Advs, dtype=np.float32)
        # use pre-step dones here
        dones = np.array(self.dones[:-1], dtype=bool)
        self.reset(self.dones[-1])
        return obs, acts, dones, Rs, Advs


class ReplayBuffer(TransBuffer):
    def __init__(self, buffer_size, batch_size):
        self.buffer_size = buffer_size
        self.batch_size = batch_size
        self.cum_size = 0
        self.buffer = []

    def add_transition(self, ob, a, r, next_ob, done):
        experience = (ob, a, r, next_ob, done)
        if self.cum_size < self.buffer_size:
            self.buffer.append(experience)
        else:
            ind = int(self.cum_size % self.buffer_size)
            self.buffer[ind] = experience
        self.cum_size += 1

    def reset(self):
        self.buffer = []
        self.cum_size = 0

    def sample_transition(self):
        # Randomly sample batch_size examples
        minibatch = random.sample(self.buffer, self.batch_size)
        state_batch = np.asarray([data[0] for data in minibatch])
        action_batch = np.asarray([data[1] for data in minibatch])
        next_state_batch = np.asarray([data[3] for data in minibatch])
        reward_batch = np.asarray([data[2] for data in minibatch])
        done_batch = np.asarray([data[4] for data in minibatch])
        return state_batch, action_batch, next_state_batch, reward_batch, done_batch

    @property
    def size(self):
        return min(self.buffer_size, self.cum_size)

class Scheduler:
    def __init__(self, val_init, val_min=0, total_step=0, decay='linear'):
        self.val = val_init
        self.N = float(total_step)
        self.val_min = val_min
        self.decay = decay
        self.decay = decay
        self.n = 0

    def get(self, n_step):
        self.n += n_step
        if self.decay == 'linear':
            return max(self.val_min, self.val * (1 - self.n / self.N))
        else:
            return self.val


if __name__ == '__main__':
    test_layers()



Instructions for updating:
non-resource variables are not supported in the long term
2.21.0
fc
(10, 10)
0.03693473 0.4456858 -1.0263959 0.796174
(10,)
0.0 0.0 0.0 0.0
lstm
(2, 40)
0.008436771 0.22344759 -0.45460182 0.50888216
(10, 40)
0.00953397 0.22340345 -0.74394757 0.6340339
(40,)
0.0 0.0 0.0 0.0
conv1
(10,)
0.0 0.0 0.0 0.0
(4, 1, 10)
-0.06959409 0.4417654 -0.7587998 0.84212744
conv2
(10,)
0.0 0.0 0.0 0.0
(4, 4, 1, 10)
-0.0336613 0.3519473 -1.1486382 0.90310764
fc
(5, 10)
lstm_done
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
lstm
[[ 0.27464956  0.27012336  0.14754139  0.05251218  0.0560856  -0.04247548
   0.31839067  0.20877455  0.21552582 -0.29177374]
 [ 0.11870658  0.20815718  0.04615565  0.12011941  0.03352695  0.01324103
   0.29941198  0.0201524   0.11516014 -0.12810265]
 [ 0.00087431 -0.01933189 

## 9. Agent Policies (`agents/policies.py`)

Actor-critic policy networks. The hierarchy is:

```
ACPolicy                  ← base: loss, optimizer, summary
  └─ LstmACPolicy         ← LSTM actor + LSTM critic (separate nets)
       └─ FPLstmACPolicy  ← adds fingerprint branch (neighbour policy hints)
            └─ FPLstmACPolicySR  ← signal+route variant with 4 input branches
                                   (wave, wait, other-agent state, fingerprint)
FcACPolicy                ← fully-connected (no LSTM, no fingerprint)
  └─ FPFcACPolicy         ← FC with fingerprint branch
QPolicy / DeepQPolicy / LRQPolicy  ← Q-learning variants (not used in MA2C)
```

**Batched forward/backward:**
`LstmACPolicy` exposes `prepare_forward / consume_forward` and
`prepare_backward / consume_backward` so the parent `IA2C` model can fold
all N agents' TF ops into a single `sess.run`, cutting round-trip overhead
from O(N) to O(1).


In [12]:
import numpy as np
from agents.utils import *
import bisect
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

class ACPolicy:
    def __init__(self, n_a, n_s, n_step, policy_name, agent_name):
        self.name = policy_name
        if agent_name is not None:
            self.name += '_' + str(agent_name)
        self.n_a = n_a
        self.n_s = n_s
        self.n_step = n_step

    def forward(self, ob, *_args, **_kwargs):
        raise NotImplementedError()

    def _build_out_net(self, h, out_type):
        if out_type == 'pi':
            pi = fc(h, out_type, self.n_a, act=tf.nn.softmax)
            return tf.squeeze(pi)
        else:
            v = fc(h, out_type, 1, act=lambda x: x)
            return tf.squeeze(v)

    def _get_forward_outs(self, out_type):
        outs = []
        if 'p' in out_type:
            outs.append(self.pi)
        if 'v' in out_type:
            outs.append(self.v)
        return outs

    def _return_forward_outs(self, out_values):
        if len(out_values) == 1:
            return out_values[0]
        return out_values

    def prepare_loss(self,max_grad_norm, adam_beta1, adam_beta2, adam_epsilon):
        self.A = tf.placeholder(tf.int32, [self.n_step])
        self.ADV = tf.placeholder(tf.float32, [self.n_step])
        self.R = tf.placeholder(tf.float32, [self.n_step])
        A_sparse = tf.one_hot(self.A, self.n_a)
        log_pi = tf.log(tf.clip_by_value(self.pi, 1e-10, 1.0))
        entropy = -tf.reduce_sum(self.pi * log_pi, axis=1)
        entropy_loss = -tf.reduce_mean(entropy)
        policy_loss = -tf.reduce_mean(tf.reduce_sum(log_pi * A_sparse, axis=1) * self.ADV)
        value_loss = tf.reduce_mean(tf.square(self.R - self.v)) * 0.5
        self.loss = policy_loss + value_loss + entropy_loss

        self.lr = tf.placeholder(tf.float32, [])

        wts = tf.trainable_variables(scope=self.name)
        grads = tf.gradients(policy_loss+ entropy_loss*self.entropy_coef, wts)
        if max_grad_norm > 0:
            grads, self.grad_norm = tf.clip_by_global_norm(grads, max_grad_norm)
        self.actor_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr, beta1=adam_beta1, beta2=adam_beta2, epsilon=adam_epsilon)
        self.actor_train = self.actor_optimizer.apply_gradients(list(zip(grads, wts)))

        wts = tf.trainable_variables(scope=self.name)
        grads = tf.gradients(value_loss, wts)
        if max_grad_norm > 0:
            grads, self.grad_norm = tf.clip_by_global_norm(grads, max_grad_norm)
        self.critic_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr, beta1=adam_beta1, beta2=adam_beta2, epsilon=adam_epsilon)
        self.critic_train = self.critic_optimizer.apply_gradients(list(zip(grads, wts)))


        # monitor training
        if self.name.endswith('_0a'):
            summaries = []
            summaries.append(tf.summary.scalar('loss/%s_policy_loss' % self.name, policy_loss))
            summaries.append(tf.summary.scalar('loss/%s_value_loss' % self.name, value_loss))
            summaries.append(tf.summary.scalar('loss/%s_total_loss' % self.name, self.loss))
            summaries.append(tf.summary.scalar('train/%s_gradnorm' % self.name, self.grad_norm))
            self.summary = tf.summary.merge(summaries)


class LstmACPolicy(ACPolicy):
    def __init__(self, n_s, n_a, n_w, n_step, n_fc_wave=128, n_fc_wait=32,n_lstm=64, name=None):
        super().__init__(n_a, n_s, n_step, 'lstm', name)
        self.n_lstm = n_lstm
        self.n_fc_wait = n_fc_wait
        self.n_fc_wave = n_fc_wave

        self.n_w = n_w
        self.ob_fw = tf.placeholder(tf.float32, [1, n_s + n_w]) # forward 1-step
        self.done_fw = tf.placeholder(tf.float32, [1])
        self.ob_bw = tf.placeholder(tf.float32, [n_step, n_s + n_w]) # backward n-step
        self.done_bw = tf.placeholder(tf.float32, [n_step])
        self.states = tf.placeholder(tf.float32, [2, n_lstm])
        with tf.variable_scope(self.name):
            # pi and v use separate nets
            self.pi_fw, pi_state = self._build_net('forward', 'pi')
            self.v_fw, v_state = self._build_net('forward', 'v')
            pi_state = tf.expand_dims(pi_state, 0)
            v_state = tf.expand_dims(v_state, 0)
            self.new_states = tf.concat([pi_state, v_state], 0)
        with tf.variable_scope(self.name, reuse=True):
            self.pi, _ = self._build_net('backward', 'pi')
            self.v, _ = self._build_net('backward', 'v')
        self._reset()

    def _build_net(self, in_type, out_type):
        if in_type == 'forward':
            ob = self.ob_fw
            done = self.done_fw
        else:
            ob = self.ob_bw
            done = self.done_bw
        if out_type == 'pi':
            states = self.states[0]
        else:
            states = self.states[1]
        if self.n_w == 0:
            h = fc(ob, out_type + '_fcw', self.n_fc_wave)
        else:
            h0 = fc(ob[:, :self.n_s], out_type + '_fcw', self.n_fc_wave)
            h1 = fc(ob[:, self.n_s:], out_type + '_fct', self.n_fc_wait)
            h = tf.concat([h0, h1], 1)
        #h = tf.concat([h0, h1, h2, h3], 1)
        h, new_states = lstm(h, done, states, out_type + '_lstm')
        out_val = self._build_out_net(h, out_type)
        '''
        #h=fc(h,out_type+'_fcs',self.n_fc_second)
        h, states = lstm(h, done, states, out_type + '_lstm1')
        h, new_states = lstm(h, done, states, out_type + '_lstm2')
        out_val = self._build_out_net(h, out_type)
        '''
        return out_val, new_states

    def _reset(self):
        # forget the cumulative states every cum_step
        self.states_fw = np.zeros((2, self.n_lstm), dtype=np.float32)
        self.states_bw = np.zeros((2, self.n_lstm), dtype=np.float32)

    # ---- Split forward/backward into prepare_/consume_ halves so a parent
    # ---- model (e.g. MA2C) can fold N agents' fetches+feeds into a single
    # ---- sess.run, instead of paying N round-trips and N first-call TF
    # ---- graph-compile costs.

    def prepare_forward(self, ob, done, out_type='pv'):
        """Build the (fetches, feeds, has_state_tail) tuple this agent needs
        for one forward step. Does NOT call sess.run."""
        fetches = self._get_forward_outs(out_type)
        has_state_tail = 'p' in out_type
        if has_state_tail:
            fetches = list(fetches)  # avoid mutating the cached list
            fetches.append(self.new_states)
        feeds = {
            self.ob_fw: np.array([ob]),
            self.done_fw: np.array([done]),
            self.states: self.states_fw,
        }
        return fetches, feeds, has_state_tail

    def consume_forward(self, out_values, out_type='pv', has_state_tail=None):
        """Apply one agent's forward results: update LSTM state, format
        return value the same way the old forward() did."""
        if has_state_tail is None:
            has_state_tail = 'p' in out_type
        if has_state_tail:
            self.states_fw = out_values[-1]
            out_values = out_values[:-1]
        return self._return_forward_outs(out_values)

    def forward(self, sess, ob, done, out_type='pv'):
        # Backwards-compatible single-agent path. Kept so non-MA2C callers
        # (e.g. evaluator) keep working.
        fetches, feeds, has_state_tail = self.prepare_forward(ob, done, out_type)
        out_values = sess.run(fetches, feeds)
        return self.consume_forward(out_values, out_type, has_state_tail)

    def prepare_backward(self, obs, acts, dones, Rs, Advs, cur_lr,
                         summary_writer=None):
        """Build the (ops, feeds) tuple this agent needs for one backward
        step. Caller is responsible for running it and then handing the
        result back to consume_backward()."""
        if summary_writer is None:
            ops = [self.actor_train, self.critic_train]
        else:
            # Only the agent built with name '...0a' has self.summary; see
            # ACPolicy.prepare_loss. Caller must only pass summary_writer
            # to that agent.
            ops = [self.summary, self.actor_train, self.critic_train]
        feeds = {
            self.ob_bw: obs,
            self.done_bw: dones,
            self.states: self.states_bw,
            self.A: acts,
            self.ADV: Advs,
            self.R: Rs,
            self.lr: cur_lr,
        }
        return ops, feeds

    def consume_backward(self, outs, summary_writer=None, global_step=None):
        self.states_bw = np.copy(self.states_fw)
        if summary_writer is not None:
            summary_writer.add_summary(outs[0], global_step=global_step)

    def backward(self, sess, obs, acts, dones, Rs, Advs, cur_lr,
                 summary_writer=None, global_step=None):
        # Backwards-compatible single-agent path.
        ops, feeds = self.prepare_backward(obs, acts, dones, Rs, Advs, cur_lr,
                                           summary_writer=summary_writer)
        outs = sess.run(ops, feeds)
        self.consume_backward(outs, summary_writer=summary_writer,
                              global_step=global_step)

    def _get_forward_outs(self, out_type):
        outs = []
        if 'p' in out_type:
            outs.append(self.pi_fw)
        if 'v' in out_type:
            outs.append(self.v_fw)
        return outs


class FPLstmACPolicy(LstmACPolicy):
    def __init__(self, n_s, n_a, n_w, n_f, n_step, n_fc_wave=128, n_fc_wait=32, n_fc_fp=32,n_lstm=64, name=None,entropy_coef=0):
        ACPolicy.__init__(self, n_a, n_s, n_step, 'fplstm', name)
        self.n_lstm = n_lstm
        self.n_fc_wave = n_fc_wave
        self.n_fc_wait = n_fc_wait
        self.n_fc_fp = n_fc_fp
        self.n_w = n_w
        self.ob_fw = tf.placeholder(tf.float32, [1, n_s + n_w + n_f]) # forward 1-step
        self.done_fw = tf.placeholder(tf.float32, [1])
        self.ob_bw = tf.placeholder(tf.float32, [n_step, n_s + n_w + n_f]) # backward n-step
        self.done_bw = tf.placeholder(tf.float32, [n_step])
        self.states = tf.placeholder(tf.float32, [2, n_lstm])
        self.entropy_coef=entropy_coef
        with tf.variable_scope(self.name):
            # pi and v use separate nets
            self.pi_fw, pi_state = self._build_net('forward', 'pi')
            self.v_fw, v_state = self._build_net('forward', 'v')
            pi_state = tf.expand_dims(pi_state, 0)
            v_state = tf.expand_dims(v_state, 0)
            self.new_states = tf.concat([pi_state, v_state], 0)
        with tf.variable_scope(self.name, reuse=True):
            self.pi, _ = self._build_net('backward', 'pi')
            self.v, _ = self._build_net('backward', 'v')
        self._reset()

    def _build_net(self, in_type, out_type):
        if in_type == 'forward':
            ob = self.ob_fw
            done = self.done_fw
        else:
            ob = self.ob_bw
            done = self.done_bw
        if out_type == 'pi':
            states = self.states[0]
        else:
            states = self.states[1]
        h0 = fc(ob[:, :self.n_s], out_type + '_fcw', self.n_fc_wave)
        h1 = fc(ob[:, (self.n_s + self.n_w):], out_type + '_fcf', self.n_fc_fp)
        if self.n_w == 0:
            h = tf.concat([h0, h1], 1)
        else:
            h2 = fc(ob[:, self.n_s: (self.n_s + self.n_w)], out_type + '_fct', self.n_fc_wait)
            h = tf.concat([h0, h2, h1], 1)
        h, new_states = lstm(h, done, states, out_type + '_lstm')
        out_val = self._build_out_net(h, out_type)

        return out_val, new_states


class FPLstmACPolicySR(FPLstmACPolicy):
    def __init__(self, n_s, n_a, n_w, n_o,n_f, n_step, n_fc_wave=128, n_fc_wait=32,n_fc_other=0, n_fc_fp=32, n_lstm=64, name=None,entropy_coef=0):
        ACPolicy.__init__(self, n_a, n_s, n_step, 'fplstm', name)
        self.n_lstm = n_lstm
        self.n_fc_wave = n_fc_wave
        self.n_fc_wait = n_fc_wait
        self.n_fc_other=n_fc_other
        self.n_fc_fp = n_fc_fp
        self.n_w = n_w
        self.n_o=n_o
        self.ob_fw = tf.placeholder(tf.float32, [1, n_s + n_w +n_o+ n_f]) # forward 1-step
        self.done_fw = tf.placeholder(tf.float32, [1])
        self.ob_bw = tf.placeholder(tf.float32, [n_step, n_s + n_w +n_o+ n_f]) # backward n-step
        self.done_bw = tf.placeholder(tf.float32, [n_step])
        self.states = tf.placeholder(tf.float32, [2, n_lstm])
        self.entropy_coef=entropy_coef
        with tf.variable_scope(self.name):
            # pi and v use separate nets
            self.pi_fw, pi_state = self._build_net('forward', 'pi')
            self.v_fw, v_state = self._build_net('forward', 'v')
            pi_state = tf.expand_dims(pi_state, 0)
            v_state = tf.expand_dims(v_state, 0)
            self.new_states = tf.concat([pi_state, v_state], 0)
        with tf.variable_scope(self.name, reuse=True):
            self.pi, _ = self._build_net('backward', 'pi')
            self.v, _ = self._build_net('backward', 'v')
        self._reset()

    def _build_net(self, in_type, out_type):
        if in_type == 'forward':
            ob = self.ob_fw
            done = self.done_fw
        else:
            ob = self.ob_bw
            done = self.done_bw
        if out_type == 'pi':
            states = self.states[0]
        else:
            states = self.states[1]

        h0 = fc(ob[:, :self.n_s], out_type + '_fcw', self.n_fc_wave)
        h1 = fc(ob[:, self.n_s: (self.n_s + self.n_w)], out_type + '_fct', self.n_fc_wait)
        h2 = fc(ob[:, (self.n_s + self.n_w):(self.n_s + self.n_w+self.n_o)], out_type + '_fco', self.n_fc_other)
        h3 = fc(ob[:, (self.n_s + self.n_w+self.n_o):], out_type + '_fcf', self.n_fc_fp)

        h = tf.concat([h0, h1, h2, h3], 1)
        h, new_states = lstm(h, done, states, out_type + '_lstm')
        out_val = self._build_out_net(h, out_type)

        return out_val, new_states

class FcACPolicy(ACPolicy):
    def __init__(self, n_s, n_a, n_w, n_step, n_fc_wave=128, n_fc_wait=32, n_lstm=64, name=None):
        super().__init__(n_a, n_s, n_step, 'fc', name)
        self.n_fc_wave = n_fc_wave
        self.n_fc_wait = n_fc_wait
        self.n_fc = n_lstm
        self.n_w = n_w
        self.obs = tf.placeholder(tf.float32, [None, n_s + n_w])
        with tf.variable_scope(self.name):
            # pi and v use separate nets
            self.pi = self._build_net('pi')
            self.v = self._build_net('v')

    def _build_net(self, out_type):
        if self.n_w == 0:
            h = fc(self.obs, out_type + '_fcw', self.n_fc_wave)
        else:
            h0 = fc(self.obs[:, :self.n_s], out_type + '_fcw', self.n_fc_wave)
            h1 = fc(self.obs[:, self.n_s:], out_type + '_fct', self.n_fc_wait)
            h = tf.concat([h0, h1], 1)
        h = fc(h, out_type + '_fc', self.n_fc)
        return self._build_out_net(h, out_type)

    def forward(self, sess, ob, done, out_type='pv'):
        outs = self._get_forward_outs(out_type)
        out_values = sess.run(outs, {self.obs: np.array([ob])})
        return self._return_forward_outs(out_values)

    def backward(self, sess, obs, acts, dones, Rs, Advs, cur_lr,
                 summary_writer=None, global_step=None):
        if summary_writer is None:
            ops = [self.actor_train,self.critic_train]
        else:
            ops = [self.summary, self.actor_train,self.critic_train]
        outs = sess.run(ops,
                        {self.obs: obs,
                         self.A: acts,
                         self.ADV: Advs,
                         self.R: Rs,
                         self.lr: cur_lr})
        if summary_writer is not None:
            summary_writer.add_summary(outs[0], global_step=global_step)


class FPFcACPolicy(FcACPolicy):
    def __init__(self, n_s, n_a, n_w, n_f, n_step, n_fc_wave=128, n_fc_wait=32, n_fc_fp=32, n_lstm=64, name=None):
        ACPolicy.__init__(self, n_a, n_s, n_step, 'fpfc', name)
        self.n_fc_wave = n_fc_wave
        self.n_fc_wait = n_fc_wait
        self.n_fc_fp = n_fc_fp
        self.n_fc = n_lstm
        self.n_w = n_w
        self.obs = tf.placeholder(tf.float32, [None, n_s + n_w + n_f])
        with tf.variable_scope(self.name):
            # pi and v use separate nets
            self.pi = self._build_net('pi')
            self.v = self._build_net('v')

    def _build_net(self, out_type):
        h0 = fc(self.obs[:, :self.n_s], out_type + '_fcw', self.n_fc_wave)
        h1 = fc(self.obs[:, (self.n_s + self.n_w):], out_type + '_fcf', self.n_fc_fp)
        if self.n_w == 0:
            h = tf.concat([h0, h1], 1)
        else:
            h2 = fc(self.obs[:, self.n_s: (self.n_s + self.n_w)], out_type + '_fct', self.n_fc_wait)
            h = tf.concat([h0, h1, h2], 1)
        h = fc(h, out_type + '_fc', self.n_fc)
        return self._build_out_net(h, out_type)


class QPolicy:
    def __init__(self, n_a, n_s, n_step, policy_name, agent_name):
        self.name = policy_name
        if agent_name is not None:
            # for multi-agent system
            self.name += '_' + str(agent_name)
        self.n_a = n_a
        self.n_s = n_s
        self.n_step = n_step

    def forward(self, ob, *_args, **_kwargs):
        raise NotImplementedError()

    def _build_fc_net(self, h, n_fc_ls):
        for i, n_fc in enumerate(n_fc_ls):
            h = fc(h, 'q_fc_%d' % i, n_fc)
        q = fc(h, 'q', self.n_a, act=lambda x: x)
        return tf.squeeze(q)

    def _build_net(self):
        raise NotImplementedError()

    def prepare_loss(self, max_grad_norm, gamma):
        self.A = tf.placeholder(tf.int32, [self.n_step])
        self.S1 = tf.placeholder(tf.float32, [self.n_step, self.n_s + self.n_w])
        self.R = tf.placeholder(tf.float32, [self.n_step])
        self.DONE = tf.placeholder(tf.bool, [self.n_step])
        A_sparse = tf.one_hot(self.A, self.n_a)

        # backward
        with tf.variable_scope(self.name + '_q', reuse=True):
            q0s = self._build_net(self.S)
            q0 = tf.reduce_sum(q0s * A_sparse, axis=1)
        with tf.variable_scope(self.name + '_q', reuse=True):
            q1s = self._build_net(self.S1)
            q1 = tf.reduce_max(q1s, axis=1)
        tq = tf.stop_gradient(tf.where(self.DONE, self.R, self.R + gamma * q1))
        self.loss = tf.reduce_mean(tf.square(q0 - tq))

        wts = tf.trainable_variables(scope=self.name)
        grads = tf.gradients(self.loss, wts)
        if max_grad_norm > 0:
            grads, self.grad_norm = tf.clip_by_global_norm(grads, max_grad_norm)
        self.lr = tf.placeholder(tf.float32, [])
        self.optimizer = tf.train.AdamOptimizer(learning_rate=self.lr)
        self._train = self.optimizer.apply_gradients(list(zip(grads, wts)))
        # monitor training
        if self.name.endswith('_0a'):
            summaries = []
            summaries.append(tf.summary.scalar('train/%s_loss' % self.name, self.loss))
            summaries.append(tf.summary.scalar('train/%s_q' % self.name, tf.reduce_mean(q0)))
            summaries.append(tf.summary.scalar('train/%s_tq' % self.name, tf.reduce_mean(tq)))
            summaries.append(tf.summary.scalar('train/%s_gradnorm' % self.name, self.grad_norm))
            self.summary = tf.summary.merge(summaries)


class DeepQPolicy(QPolicy):
    def __init__(self, n_s, n_a, n_w, n_step, n_fc0=128, n_fc=64, name=None):
        super().__init__(n_a, n_s, n_step, 'dqn', name)
        self.n_fc = n_fc
        self.n_fc0 = n_fc0
        self.n_w = n_w
        self.S = tf.placeholder(tf.float32, [None, n_s + n_w])
        with tf.variable_scope(self.name + '_q'):
            self.qvalues = self._build_net(self.S)

    def _build_net(self, S):
        if self.n_w == 0:
            h = fc(S, 'q_fcw', self.n_fc0)
        else:
            h0 = fc(S[:, :self.n_s], 'q_fcw', self.n_fc0)
            h1 = fc(S[:, self.n_s:], 'q_fct', self.n_fc0 / 4)
            h = tf.concat([h0, h1], 1)
        return self._build_fc_net(h, [self.n_fc])

    def forward(self, sess, ob):
        return sess.run(self.qvalues, {self.S: np.array([ob])})

    def backward(self, sess, obs, acts, next_obs, dones, rs, cur_lr,
                 summary_writer=None, global_step=None):
        if summary_writer is None:
            ops = self._train
        else:
            ops = [self.summary, self._train]
        outs = sess.run(ops,
                        {self.S: obs,
                         self.A: acts,
                         self.S1: next_obs,
                         self.DONE: dones,
                         self.R: rs,
                         self.lr: cur_lr})
        if summary_writer is not None:
            summary_writer.add_summary(outs[0], global_step=global_step)


class LRQPolicy(DeepQPolicy):
    def __init__(self, n_s, n_a, n_step, name=None):
        QPolicy.__init__(self, n_a, n_s, n_step, 'lr', name)
        self.S = tf.placeholder(tf.float32, [None, n_s])
        self.n_w = 0
        with tf.variable_scope(self.name + '_q'):
            self.qvalues = self._build_net(self.S)

    def _build_net(self, S):
        return self._build_fc_net(S, [])


## 10. Agent Models (`agents/models.py`)

Three classes with increasing coordination:

| Class | Description |
|-------|-------------|
| `A2C` | Single-agent advantage actor-critic |
| `IA2C(A2C)` | Independent MA2C — each agent has its own policy, no shared parameters |
| `MA2C(IA2C)` | Full multi-agent A2C — uses fingerprints so each agent can condition on neighbours' policies |

`MA2C.__init__` instantiates one `FPLstmACPolicySR` per agent (signal or
route type) inside a single shared TF graph and session.

`IA2C.forward` and `IA2C.backward` use chunked batching (`_BATCH_CHUNK = 20`)
to balance memory vs. round-trip cost during TF graph compilation.


In [13]:
import os
import sys
import time
from agents.utils import *
from agents.policies import *
import logging
import multiprocessing as mp
import numpy as np
#import tensorflow as tf
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# Counters used by MA2C.forward / IA2C.backward to gate verbose progress
# logging — TF first-call graph optimization can take many seconds per
# subgraph on CPU, and with ~192 agents the loop looks dead from the
# outside. Prints fade out after the first few cycles.
_FORWARD_CALL_COUNT = 0
_BACKWARD_CALL_COUNT = 0

# How many agents to fold into a single sess.run. Lower = less memory
# pressure during TF Grappler optimization, more Python<->TF round-trips.
# Higher = the opposite. Override via env var MA2C_BATCH_CHUNK if you
# want to tune.
_BATCH_CHUNK = int(os.environ.get('MA2C_BATCH_CHUNK', '20'))

class A2C:
    def __init__(self, n_s, n_a, total_step, model_config, seed=0, n_f=None):
        # load parameters
        self.name = 'a2c'
        self.n_agent = 1
        self.problem =None

        self.n_s = n_s
        self.n_a = n_a
        self.n_step = model_config.getint('batch_size')
        # init tf
        tf.reset_default_graph()
        tf.set_random_seed(seed)
        _n_cores = os.cpu_count() or 8
        config = tf.ConfigProto(
            allow_soft_placement=True,
            intra_op_parallelism_threads=_n_cores,
            inter_op_parallelism_threads=max(2, _n_cores // 2),
        )
        # Let TF allocate GPU memory on demand if a CUDA device is present.
        config.gpu_options.allow_growth = True
        self.sess = tf.Session(config=config)
        self.policy = self._init_policy(n_s, n_a, n_f, model_config)
        self.saver = tf.train.Saver(max_to_keep=5)
        if total_step:
            # training
            self.total_step = total_step
            self._init_scheduler(model_config)
            self._init_train(model_config)
        self.sess.run(tf.global_variables_initializer())

    def _init_policy(self, n_s, n_a, n_w, n_f, model_config,n_o=0,agent_type=None, agent_name=None):
        n_lstm = model_config.getint('num_lstm')
        if self.problem=='signal':
            n_fw_s = model_config.getint('s_num_fw')
            n_ft_s = model_config.getint('s_num_ft')
            n_fp_s = model_config.getint('s_num_fp')
            entropy=model_config.getfloat('entropy_coef_s')
            policy = FPLstmACPolicy(n_s, n_a, n_w, n_f, self.n_step, n_fc_wave=n_fw_s,
                                    n_fc_wait=n_ft_s, n_fc_fp=n_fp_s,n_lstm=n_lstm, name=agent_name,entropy_coef=entropy)
        elif self.problem == 'route':
            n_fw_r = model_config.getint('r_num_fw')
            n_ft_r = model_config.getint('r_num_ft')
            n_fp_r = model_config.getint('r_num_fp')
            entropy = model_config.getfloat('entropy_coef_r')
            policy = FPLstmACPolicy(n_s, n_a, n_w, n_f, self.n_step, n_fc_wave=n_fw_r,
                                    n_fc_wait=n_ft_r, n_fc_fp=n_fp_r, n_lstm=n_lstm, name=agent_name,entropy_coef=entropy)
        if self.problem=='signal_route':
            n_fw_s = model_config.getint('num_fw_s')
            n_ft_s = model_config.getint('num_ft_s')
            n_fo_s = model_config.getint('num_fo_s')
            n_fp_s = model_config.getint('num_fp_s')

            n_fw_r = model_config.getint('num_fw_r')
            n_ft_r = model_config.getint('num_ft_r')
            n_fo_r = model_config.getint('num_fo_r')
            n_fp_r = model_config.getint('num_fp_r')
            entropy_s = model_config.getfloat('entropy_coef_s')
            entropy_r = model_config.getfloat('entropy_coef_r')
            if agent_type=='signal':
                #agent=signal
                policy = FPLstmACPolicySR(n_s, n_a, n_w,n_o, n_f, self.n_step, n_fc_wave=n_fw_s,
                                    n_fc_wait=n_ft_s, n_fc_other=n_fo_s,n_fc_fp=n_fp_s,n_lstm=n_lstm, name=agent_name,entropy_coef=entropy_s)
            else:
                #agent=route
                policy = FPLstmACPolicySR(n_s, n_a, n_w, n_o, n_f, self.n_step, n_fc_wave=n_fw_r,
                                         n_fc_wait=n_ft_r, n_fc_other=n_fo_r, n_fc_fp=n_fp_r,n_lstm=n_lstm, name=agent_name,entropy_coef=entropy_r)

        if self.name != 'ma2c':
            policy = LstmACPolicy(n_s, n_a, n_w, self.n_step, n_fc_wave=n_fw_s,
                                  n_fc_wait=n_ft_s,n_lstm=n_lstm, name=agent_name)
        return policy

    def _init_scheduler(self, model_config):
        lr_init = model_config.getfloat('lr_init')
        lr_decay = model_config.get('lr_decay')

        if lr_decay == 'constant':
            self.lr_scheduler = Scheduler(lr_init, decay=lr_decay)
        else:
            lr_min = model_config.getfloat('LR_MIN')
            self.lr_scheduler = Scheduler(lr_init, lr_min, self.total_step, decay=lr_decay)


    def _init_train(self, model_config):
        # init loss
        entropy_coef=model_config.getfloat('entropy_coef')
        max_grad_norm = model_config.getfloat('max_grad_norm')
        adam_beta1=model_config.getfloat('adam_beta1')
        adam_beta2=model_config.getfloat('adam_beta2')
        adam_epsilon = model_config.getfloat('adam_epsilon')
        self.policy.prepare_loss(max_grad_norm, adam_beta1,adam_beta2, adam_epsilon)

        # init replay buffer
        gamma = model_config.getfloat('gamma')
        self.trans_buffer = OnPolicyBuffer(gamma)

    def save(self, model_dir, global_step):
        self.saver.save(self.sess, model_dir + 'checkpoint', global_step=global_step)

    def load(self, model_dir, checkpoint=None):
        save_file = None
        save_step = 0
        if os.path.exists(model_dir):
            if checkpoint is None:
                for file in os.listdir(model_dir):
                    if file.startswith('checkpoint'):
                        prefix = file.split('.')[0]
                        tokens = prefix.split('-')
                        if len(tokens) != 2:
                            continue
                        cur_step = int(tokens[1])
                        if cur_step > save_step:
                            save_file = prefix
                            save_step = cur_step
            else:
                save_file = 'checkpoint-' + str(int(checkpoint))
        if save_file is not None:
            self.saver.restore(self.sess, model_dir + save_file)
            logging.info('Checkpoint loaded: %s' % save_file)
            return True
        logging.error('Can not find old checkpoint for %s' % model_dir)
        return False

    def reset(self):
        self.policy._reset()

    def backward(self, R, summary_writer=None, global_step=None):
        cur_lr = self.lr_scheduler.get(self.n_step)
        obs, acts, dones, Rs, Advs = self.trans_buffer.sample_transition(R)
        self.policy.backward(self.sess, obs, acts, dones, Rs, Advs, cur_lr,
                             summary_writer=summary_writer, global_step=global_step)

    def forward(self, ob, done, out_type='pv'):
        return self.policy.forward(self.sess, ob, done, out_type)

    def add_transition(self, ob, action, reward, value, done):
        # Hard code the reward norm for negative reward only
        self.trans_buffer.add_transition(ob, action, reward, value, done)

class IA2C(A2C):
    def __init__(self, n_s_ls, n_a_ls, n_w_ls, total_step,
                 model_config, seed=0):
        self.name = 'ia2c'
        self.agents = []
        self.n_agent = len(n_s_ls)

        self.n_s_ls = n_s_ls
        self.n_a_ls = n_a_ls
        self.n_w_ls = n_w_ls
        self.n_step = model_config.getint('batch_size')
        # init tf
        tf.reset_default_graph()
        tf.set_random_seed(seed)
        _n_cores = os.cpu_count() or 8
        config = tf.ConfigProto(
            allow_soft_placement=True,
            intra_op_parallelism_threads=_n_cores,
            inter_op_parallelism_threads=max(2, _n_cores // 2),
        )
        # Let TF allocate GPU memory on demand if a CUDA device is present.
        config.gpu_options.allow_growth = True
        self.sess = tf.Session(config=config)
        self.policy_ls = []
        for i, (n_s, n_w, n_a) in enumerate(zip(self.n_s_ls, self.n_w_ls, self.n_a_ls)):
            # agent_name is needed to differentiate multi-agents
            self.policy_ls.append(self._init_policy(n_s - n_w, n_a, n_w, 0, model_config,
                                  agent_name='{:d}a'.format(i)))
        self.saver = tf.train.Saver(max_to_keep=5)
        if total_step:
            # training
            self.total_step = total_step
            self._init_scheduler(model_config)
            self._init_train(model_config)
        self.sess.run(tf.global_variables_initializer())

    def _init_train(self, model_config):
        # init loss
        max_grad_norm = model_config.getfloat('max_grad_norm')
        entropy_coef=model_config.getfloat('entropy_coef')
        adam_beta1 = model_config.getfloat('adam_beta1')
        adam_beta2 = model_config.getfloat('adam_beta2')
        adam_epsilon = model_config.getfloat('adam_epsilon')

        gamma = model_config.getfloat('gamma')
        self.trans_buffer_ls = []
        for i in range(self.n_agent):
            self.policy_ls[i].prepare_loss(max_grad_norm, adam_beta1, adam_beta2, adam_epsilon)
            self.trans_buffer_ls.append(OnPolicyBuffer(gamma))

    def backward(self, R_ls, summary_writer=None, global_step=None):
        """Chunked-batched backward: process _BATCH_CHUNK agents per
        sess.run instead of all N. Bounds TF Grappler peak memory while
        still saving most of the per-call Python<->TF overhead."""
        global _BACKWARD_CALL_COUNT
        _BACKWARD_CALL_COUNT += 1
        verbose = _BACKWARD_CALL_COUNT <= 2
        bwd_t0 = time.time()
        cur_lr = self.lr_scheduler.get(self.n_step)
        n_chunks = (self.n_agent + _BATCH_CHUNK - 1) // _BATCH_CHUNK
        if verbose:
            print('  [backward call %d] %d agents in %d chunks of %d...'
                  % (_BACKWARD_CALL_COUNT, self.n_agent, n_chunks,
                     _BATCH_CHUNK), flush=True)

        for chunk_idx in range(n_chunks):
            chunk_start = chunk_idx * _BATCH_CHUNK
            chunk_end = min(chunk_start + _BATCH_CHUNK, self.n_agent)
            chunk_t0 = time.time()

            chunk_ops = []
            chunk_feeds = {}
            ops_per_agent = []
            for i in range(chunk_start, chunk_end):
                obs, acts, dones, Rs, Advs = self.trans_buffer_ls[i].sample_transition(R_ls[i])
                # Only the first agent (i==0) has a self.summary op.
                sw = summary_writer if i == 0 else None
                ops, feeds = self.policy_ls[i].prepare_backward(
                    obs, acts, dones, Rs, Advs, cur_lr, summary_writer=sw)
                ops_per_agent.append(len(ops))
                chunk_ops.extend(ops)
                chunk_feeds.update(feeds)

            chunk_outs = self.sess.run(chunk_ops, chunk_feeds)

            idx = 0
            for k, i in enumerate(range(chunk_start, chunk_end)):
                n = ops_per_agent[k]
                agent_outs = chunk_outs[idx:idx + n]
                idx += n
                sw = summary_writer if i == 0 else None
                gs = global_step if i == 0 else None
                self.policy_ls[i].consume_backward(agent_outs, summary_writer=sw,
                                                    global_step=gs)

            if verbose:
                print('    chunk %d/%d (agents %d-%d): %.1fs'
                      % (chunk_idx + 1, n_chunks, chunk_start, chunk_end - 1,
                         time.time() - chunk_t0), flush=True)

        if verbose:
            print('  [backward call %d] DONE in %.1fs'
                  % (_BACKWARD_CALL_COUNT, time.time() - bwd_t0), flush=True)

    def forward(self, obs, done, out_type='pv'):
        """Chunked-batched forward: see backward() docstring. Same idea."""
        global _FORWARD_CALL_COUNT
        _FORWARD_CALL_COUNT += 1
        verbose = _FORWARD_CALL_COUNT <= 2
        fwd_t0 = time.time()
        n_chunks = (self.n_agent + _BATCH_CHUNK - 1) // _BATCH_CHUNK
        if verbose:
            print('  [forward call %d] %d agents in %d chunks of %d...'
                  % (_FORWARD_CALL_COUNT, self.n_agent, n_chunks,
                     _BATCH_CHUNK), flush=True)

        # Output accumulators in agent-index order
        if len(out_type) == 1:
            out = [None] * self.n_agent
        else:
            out1 = [None] * self.n_agent
            out2 = [None] * self.n_agent

        for chunk_idx in range(n_chunks):
            chunk_start = chunk_idx * _BATCH_CHUNK
            chunk_end = min(chunk_start + _BATCH_CHUNK, self.n_agent)
            chunk_t0 = time.time()

            chunk_fetches = []
            chunk_feeds = {}
            fetches_per_agent = []
            has_state_per_agent = []
            for i in range(chunk_start, chunk_end):
                fetches, feeds, has_state = self.policy_ls[i].prepare_forward(
                    obs[i], done, out_type)
                fetches_per_agent.append(len(fetches))
                has_state_per_agent.append(has_state)
                chunk_fetches.extend(fetches)
                chunk_feeds.update(feeds)

            chunk_results = self.sess.run(chunk_fetches, chunk_feeds)

            idx = 0
            for k, i in enumerate(range(chunk_start, chunk_end)):
                n = fetches_per_agent[k]
                agent_out = self.policy_ls[i].consume_forward(
                    chunk_results[idx:idx + n], out_type, has_state_per_agent[k])
                idx += n
                if len(out_type) == 1:
                    out[i] = agent_out
                else:
                    out1[i] = agent_out[0]
                    out2[i] = agent_out[1]

            if verbose:
                print('    chunk %d/%d (agents %d-%d): %.1fs'
                      % (chunk_idx + 1, n_chunks, chunk_start, chunk_end - 1,
                         time.time() - chunk_t0), flush=True)

        if verbose:
            print('  [forward call %d] DONE in %.1fs'
                  % (_FORWARD_CALL_COUNT, time.time() - fwd_t0), flush=True)
        if len(out_type) == 1:
            return out
        else:
            return out1, out2

    def backward_mp(self, R_ls, summary_writer=None, global_step=None):
        cur_lr = self.lr_scheduler.get(self.n_step)

        def worker(i):
            obs, acts, dones, Rs, Advs = self.trans_buffer_ls[i].sample_transition(R_ls[i])
            self.policy_ls[i].backward(self.sess, obs, acts, dones, Rs, Advs, cur_lr,
                                       summary_writer=summary_writer, global_step=global_step)
        mps = []
        for i in range(self.n_agent):
            p = mp.Process(target=worker, args=(i))
            p.start()
            mps.append(p)
        for p in mps:
            p.join()

    def reset(self):
        for policy in self.policy_ls:
            policy._reset()

    def add_transition(self, obs, actions, rewards, values, done):
        for i in range(self.n_agent):
            self.trans_buffer_ls[i].add_transition(obs[i], actions[i],
                                                   rewards[i], values[i], done)

class MA2C(IA2C):
    def __init__(self, n_s_ls, n_a_ls, n_w_ls, n_f_ls,total_step,
                 model_config,problem,agent_type=None ,n_o_ls=None,seed=0):
        self.name = 'ma2c'
        self.agents = []
        self.n_agent = len(n_s_ls)
        self.problem = problem
        self.n_s_ls = n_s_ls
        self.n_a_ls = n_a_ls
        self.n_f_ls = n_f_ls
        self.n_w_ls = n_w_ls
        self.n_step = model_config.getint('batch_size')
        # init tf
        tf.reset_default_graph()
        tf.set_random_seed(seed)
        _n_cores = os.cpu_count() or 8
        config = tf.ConfigProto(
            allow_soft_placement=True,
            intra_op_parallelism_threads=_n_cores,
            inter_op_parallelism_threads=max(2, _n_cores // 2),
        )
        # Let TF allocate GPU memory on demand if a CUDA device is present.
        config.gpu_options.allow_growth = True
        self.sess = tf.Session(config=config)
        self.policy_ls = []
        if self.problem=='signal_route':
            self.n_o_ls=n_o_ls
            self.agent_type = agent_type
            for i, (n_s, n_a, n_w, n_o,n_f,a_t) in enumerate(zip(self.n_s_ls, self.n_a_ls, self.n_w_ls, self.n_o_ls,self.n_f_ls,self.agent_type)):
                # agent_name is needed to differentiate multi-agents
                self.policy_ls.append(self._init_policy(n_s, n_a, n_w,n_f, model_config,n_o=n_o,agent_type=a_t,
                                                        agent_name='{:d}a'.format(i)))
        else:
            for i, (n_s, n_a, n_w, n_f) in enumerate(zip(self.n_s_ls, self.n_a_ls, self.n_w_ls, self.n_f_ls)):
                # agent_name is needed to differentiate multi-agents
                self.policy_ls.append(self._init_policy(n_s, n_a, n_w, n_f, model_config,
                                                        agent_name='{:d}a'.format(i)))

        self.saver = tf.train.Saver(max_to_keep=5)
        if total_step:
            # training
            self.total_step = total_step
            self._init_scheduler(model_config)
            self._init_train(model_config)
        self.sess.run(tf.global_variables_initializer())

## 11. Training & Evaluation Utilities (`utils.py`)

| Class / Function | Purpose |
|-----------------|---------|
| `Counter` | Tracks global step; gates `should_test()` and `should_log()` |
| `Trainer` | Main training loop: calls `explore()` to collect n-step transitions, then `model.backward()` |
| `Tester(Trainer)` | Post-training offline testing |
| `Evaluator(Tester)` | Full evaluation with data recording; also `run_diff_com/load` for sensitivity sweeps |
| `Evaluator_separate` | Evaluates signal-only and route-only models that were trained separately |
| `plot_train()` | Reads `train_reward.csv` and saves a three-panel training curve PNG |
| `plot_evaluation()` | Bar chart comparing average wait / travel time across policies |


In [14]:
import itertools
import logging
import numpy as np
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import time
import pandas as pd


def check_dir(cur_dir):
    if not os.path.exists(cur_dir):
        return False
    return True

import os
import shutil

def copy_file(src_dir, tar_dir, suffix='.ini'):

    os.makedirs(tar_dir, exist_ok=True)

    src = src_dir
    if os.path.isdir(src_dir):
        src = find_file(src_dir, suffix)
        if src is None:
            raise FileNotFoundError(f'Cannot find {suffix} under {src_dir}')

    shutil.copy(src, tar_dir)

def find_file(cur_dir, suffix='.ini'):
    for file in os.listdir(cur_dir):
        if file.endswith(suffix):
            return cur_dir + '/' + file
    logging.error('Cannot find %s file' % suffix)
    return None


def init_dir(base_dir, pathes=['log', 'data', 'model']):
    if not os.path.exists(base_dir):
        os.mkdir(base_dir)
    dirs = {}
    for path in pathes:
        cur_dir = base_dir + '/%s/' % path
        if not os.path.exists(cur_dir):
            os.mkdir(cur_dir)
        dirs[path] = cur_dir
    return dirs


def init_log(log_dir):
    logging.basicConfig(format='%(asctime)s [%(levelname)s] %(message)s',
                        level=logging.INFO,
                        handlers=[
                            logging.FileHandler('%s/%d.log' % (log_dir, time.time())),
                            logging.StreamHandler()
                        ])


def init_test_flag(test_mode):
    if test_mode == 'no_test':
        return False, False
    if test_mode == 'in_train_test':
        return True, False
    if test_mode == 'after_train_test':
        return False, True
    if test_mode == 'all_test':
        return True, True
    return False, False


def plot_train(data_dirs, labels):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    metrics = ['reward', 'average waiting time', 'average travel time']
    ylabels = ['Cumulative Reward', 'Avg Waiting Time (s)', 'Avg Travel Time (s)']
    fig, axes = plt.subplots(len(metrics), 1, figsize=(10, 4 * len(metrics)))

    for data_dir, label in zip(data_dirs, labels):
        csv_path = os.path.join(data_dir, 'train_reward.csv')
        if not os.path.exists(csv_path):
            logging.warning('plot_train: no file at %s' % csv_path)
            continue
        df = pd.read_csv(csv_path)
        train_df = df[df['test_id'] == -1].copy()
        for ax, metric, ylabel in zip(axes, metrics, ylabels):
            ax.plot(train_df['step'], train_df[metric], label=label, linewidth=1.5)
            ax.set_xlabel('Training Step')
            ax.set_ylabel(ylabel)
            ax.set_title(ylabel)
            ax.legend()
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = os.path.join(data_dirs[0], 'train_plot.png')
    plt.savefig(out, dpi=150)
    plt.close()
    logging.info('Saved training plot to %s' % out)


def plot_evaluation(data_dirs, labels):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    records = []
    for data_dir, label in zip(data_dirs, labels):
        # try eval CSV first, fall back to train_reward.csv
        found = False
        for fname in os.listdir(data_dir):
            if fname.endswith('_trip.csv') or fname.endswith('_traffic.csv'):
                df = pd.read_csv(os.path.join(data_dir, fname))
                if 'average waiting time' in df.columns:
                    records.append({
                        'label': label,
                        'avg_wait': df['average waiting time'].mean(),
                        'avg_travel': df['average travel time'].mean(),
                    })
                    found = True
                    break
        if not found:
            csv_path = os.path.join(data_dir, 'train_reward.csv')
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                records.append({
                    'label': label,
                    'avg_wait': df['average waiting time'].mean(),
                    'avg_travel': df['average travel time'].mean(),
                })

    if not records:
        logging.warning('plot_evaluation: no data found in provided dirs')
        return

    labs = [r['label'] for r in records]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, key, title in zip(axes,
                              ['avg_wait', 'avg_travel'],
                              ['Avg Waiting Time (s)', 'Avg Travel Time (s)']):
        vals = [r[key] for r in records]
        bars = ax.bar(labs, vals, color=['steelblue', 'coral', 'seagreen'][:len(vals)])
        ax.set_title(title)
        ax.set_ylabel('Seconds')
        ax.grid(True, axis='y', alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                    '%.1f' % v, ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    out = os.path.join(data_dirs[0], 'eval_plot.png')
    plt.savefig(out, dpi=150)
    plt.close()
    logging.info('Saved evaluation plot to %s' % out)


class Counter:
    def __init__(self, total_step,test_step, log_step):
        self.counter = itertools.count(1)
        self.cur_step = 0
        self.cur_test_step = 0
        self.total_step = total_step
        self.test_step = test_step
        self.log_step = log_step
        self.stop = False

    def next(self):
        self.cur_step = next(self.counter)
        return self.cur_step

    def should_test(self):
        test = False
        if (self.cur_step - self.cur_test_step) >= self.test_step:
            test = True
            self.cur_test_step = self.cur_step
        return test

    def should_log(self):
        return (self.cur_step % self.log_step == 0)

    def should_stop(self):
        if self.cur_step >= self.total_step:
            return True
        return self.stop


class Trainer():
    def __init__(self, env, model, problem,global_counter, summary_writer, run_test, output_path=None):
        self.cur_step=0
        self.global_counter = global_counter
        self.problem=problem
        self.env = env
        self.agent = self.env.agent
        self.model = model
        self.sess = self.model.sess
        self.n_step = self.model.n_step
        self.summary_writer = summary_writer
        self.run_test = run_test
        assert self.env.T % self.n_step == 0
        self.data = []
        self.agent_reward=[]
        self.output_path = output_path
        if run_test:
            self.test_num = self.env.test_num
            logging.info('Testing: total test num: %d' % self.test_num)
        self._init_summary()

    def _init_summary(self):
        self.train_reward = tf.placeholder(tf.float32, [])
        self.train_summary = tf.summary.scalar('train_reward', self.train_reward)
        self.test_reward = tf.placeholder(tf.float32, [])
        self.test_summary = tf.summary.scalar('test_reward', self.test_reward)

    def _add_summary(self, reward, global_step, is_train=True):
        if is_train:
            summ = self.sess.run(self.train_summary, {self.train_reward: reward})
        else:
            summ = self.sess.run(self.test_summary, {self.test_reward: reward})
        self.summary_writer.add_summary(summ, global_step=global_step)

    def explore(self, prev_ob,prev_done):
        ob = prev_ob
        done = prev_done
        rewards = 0
        total_arrived_number = 0
        total_departed_number = 0
        total_waiting_time = 0
        total_travel_time = 0
        agent_reward=[]
        for i in range(self.n_step):

            policy, value = self.model.forward(ob, done)
            # need to update fingerprint before calling step
            self.env.update_fingerprint(policy)
            action = []
            for pi in policy:
                action.append(np.random.choice(np.arange(len(pi)), p=pi))
            next_ob, reward, done, global_reward,arrived_number, departed_number,waiting_time,travel_time,agent_reward0 = self.env.step(action)
            total_arrived_number += arrived_number
            total_departed_number += departed_number
            total_waiting_time += waiting_time
            total_travel_time += travel_time

            rewards += global_reward
            #if agent_reward==[]:
            if agent_reward is None or np.size(agent_reward) == 0:
                agent_reward=np.array(agent_reward0)
            else:
                agent_reward += np.array(agent_reward0)

            global_step = self.global_counter.next()
            self.cur_step +=1


            self.model.add_transition(ob, action, reward, value, done)
            if self.global_counter.should_log():
                logging.info('''Training: global step %d, current step %d, r: %.2f, train r: %.2f, done: %r''' %
                         (global_step, self.cur_step, global_reward, np.mean(reward), done))

            if done:
                break
            ob = next_ob

        if done:
            R = [0] * self.model.n_agent
        else:
            R = self.model.forward(ob, False, 'v')

        return ob, done, R, rewards,agent_reward,total_arrived_number, total_departed_number,total_waiting_time,total_travel_time

    def perform(self, test_ind, demo=True, policy_type='default',load=False):
        ob = self.env.reset(gui=demo, test_ind=test_ind,load=load)
        # note this done is pre-decision to reset LSTM states!
        done = True
        self.model.reset()
        rewards = 0
        total_arrived_number = 0
        total_departed_number = 0
        total_waiting_time = 0
        total_travel_time = 0

        total_agent_reward=[]
        while True:
            policy = self.model.forward(ob, done, 'p')
            self.env.update_fingerprint(policy)
            action = []
            for pi in policy:
                if policy_type != 'deterministic':
                    action.append(np.random.choice(np.arange(len(pi)), p=pi))
                else:
                    action.append(np.argmax(np.array(pi)))

            next_ob, reward, done, global_reward, arrived_number, departed_number,waiting_time,travel_time,agent_reward0 = self.env.step(action)

            total_arrived_number += arrived_number
            total_departed_number += departed_number
            total_waiting_time += waiting_time
            total_travel_time += travel_time

            rewards += global_reward
            if total_agent_reward is None or np.size(total_agent_reward) == 0:
                total_agent_reward=np.array(agent_reward0)
            else:
                total_agent_reward += np.array(agent_reward0)

            if done:
                break
            ob = next_ob

        return rewards,total_agent_reward,total_arrived_number, total_departed_number,total_waiting_time,total_travel_time

    def run(self):
        while not self.global_counter.should_stop():
            if self.run_test and self.global_counter.should_test():
                rewards = 0
                total_arrived_number = 0
                total_departed_number = 0
                total_waiting_time = 0
                total_travel_time = 0

                total_agent_reward=[]
                global_step = self.global_counter.cur_step
                self.env.train_mode = False
                for test_ind in range(self.test_num):
                    cur_rewards, _,arrived_number, departed_number,waiting_time,travel_time= self.perform(test_ind)
                    self.env.terminate()
                    rewards+=cur_rewards

                    total_arrived_number += arrived_number
                    total_departed_number += departed_number
                    total_waiting_time += waiting_time
                    total_travel_time += travel_time

                    log = {'agent': self.agent,
                           'step': global_step,
                           'test_id': test_ind,
                           'reward': rewards,
                            'total arrived vehicle number': total_arrived_number,
                           'total departured vehicle number': total_departed_number,
                           'total waiting time': total_waiting_time,
                           'total travel time': total_travel_time,
                           'average waiting time': total_waiting_time / total_departed_number,
                           'average travel time': total_travel_time / total_departed_number}

                    self.data.append(log)
                self._add_summary(rewards, global_step, is_train=False)
                logging.info('Testing: global step %d, avg R: %.2f' %
                             (global_step, rewards))

            # train
            self.env.train_mode = True
            ob = self.env.reset()

            done=True
            self.model.reset()

            rewards = 0
            total_arrived_number=0
            total_departed_number=0
            total_waiting_time =0
            total_travel_time = 0
            total_agent_reward=[]
            self.cur_step = 0
            while True:
                ob, done, R, cur_rewards,agent_reward,arrived_number, departed_number,waiting_time,travel_time = self.explore(ob,done)
                global_step = self.global_counter.cur_step
                rewards += cur_rewards
                total_arrived_number += arrived_number
                total_departed_number += departed_number
                total_waiting_time+=waiting_time
                total_travel_time+=travel_time

                if total_agent_reward is None or np.size(total_agent_reward) == 0:
                    total_agent_reward=np.array(agent_reward)
                else:
                    total_agent_reward+=np.array(agent_reward)

                self.model.backward(R, self.summary_writer, global_step)
                # termination
                if done:
                    #到达3600s
                    #结束 关闭SUMO文件
                    self.env.terminate()
                    break

            log = {'agent': self.agent,
                   'step': global_step,
                   'test_id': -1,
                   'reward': rewards,
                   'total arrived vehicle number':total_arrived_number,
                   'total departured vehicle number': total_departed_number,
                   'total waiting time':total_waiting_time,
                   'total travel time': total_travel_time,
                   'average waiting time':total_waiting_time/total_departed_number,
                   'average travel time': total_travel_time / total_departed_number
                   }

            self.data.append(log)
            self.agent_reward.append(total_agent_reward)

            self._add_summary(rewards, global_step)
            self.summary_writer.flush()
        df = pd.DataFrame(self.data)
        path = self.output_path.replace('data', self.problem)
        df.to_csv(path + 'train_reward.csv')
        np.savetxt(path + 'agent_reward.csv', self.agent_reward, delimiter=',')


class Tester(Trainer):
    def __init__(self, env, model,problem,global_counter, summary_writer, output_path):
        super().__init__(env, model, global_counter, summary_writer)
        self.env.train_mode = False
        self.test_num = self.env.test_num
        self.output_path = output_path
        self.problem=problem
        self.data = []
        logging.info('Testing: total test num: %d' % self.test_num)

    def _init_summary(self):
        self.reward = tf.placeholder(tf.float32, [])
        self.summary = tf.summary.scalar('test_reward', self.reward)

class Evaluator(Tester):
    def __init__(self, env, model, output_path, demo=False, policy_type='default'):
        self.env = env
        self.model = model
        self.agent = self.env.agent
        self.env.train_mode = False
        self.test_num = self.env.test_num
        self.output_path = output_path
        self.demo = demo
        self.policy_type = policy_type

    def run(self):
        is_record = True
        self.env.cur_episode = 0
        self.env.init_data(is_record, self.output_path)
        time.sleep(1)

        for test_ind in range(self.test_num):
            reward, _,_,_,_,_ = self.perform(test_ind, demo=self.demo, policy_type=self.policy_type)
            self.env.terminate()
            logging.info('test %i, avg reward %.2f' % (test_ind, reward))
            time.sleep(2)
            self.env.collect_tripinfo()
        self.env.output_data()

    def run_diff_com(self):
        numbers = np.arange(0, 1.1, 0.1)
        for i in numbers:
            self.env.CAV_rate=i

            folder_path="Sioux/%s/eva_data/%.1f/" % (self.env.problem, i)
            os.mkdir(folder_path)

            is_record = True
            self.env.cur_episode = 0
            self.env.init_data(is_record, self.output_path,diff=i)
            time.sleep(1)
            for test_ind in range(self.test_num):
                reward, _, _, _, _, _ = self.perform(test_ind, demo=False, policy_type=self.policy_type)
                self.env.terminate()
                logging.info('test %i, avg reward %.2f' % (test_ind, reward))
                time.sleep(2)
                self.env.collect_tripinfo()
            self.env.output_data()

    def run_diff_load(self):
        numbers = np.arange(0.2, 2.2, 0.2)
        for i in numbers:

            folder_path="Sioux/%s/eva_data/%.1f/" % (self.env.problem, i)
            os.mkdir(folder_path)

            is_record = True
            self.env.cur_episode = 0
            self.env.init_data(is_record, self.output_path,diff=i)
            time.sleep(1)
            for test_ind in range(self.test_num):
                reward, _, _, _, _, _ = self.perform(test_ind, demo=False, policy_type=self.policy_type,load=i)
                self.env.terminate()
                logging.info('test %i, avg reward %.2f' % (test_ind, reward))
                time.sleep(2)
                self.env.collect_tripinfo()
            self.env.output_data()

class Evaluator_separate(Evaluator):
    def __init__(self, env, model_signal,model_route, output_path, demo=False, policy_type='default'):
        self.env = env
        self.model_signal = model_signal
        self.model_route=model_route

        self.agent = self.env.agent
        self.env.train_mode = False
        self.test_num = self.env.test_num
        self.output_path = output_path
        self.demo = demo
        self.policy_type = policy_type

    def run(self):
        is_record = True
        self.env.cur_episode = 0
        self.env.init_data(is_record, self.output_path)
        time.sleep(1)

        for test_ind in range(self.test_num):
            self.perform(test_ind, demo=True, policy_type=self.policy_type)
            self.env.terminate()
            time.sleep(2)
            self.env.collect_tripinfo()
        self.env.output_data()

    def perform(self, test_ind, demo=True, policy_type='default',load=False):
        ob = self.env.reset(gui=demo, test_ind=test_ind,load=load,separate=True)
        # note this done is pre-decision to reset LSTM states!
        done = True
        self.model_signal.reset()
        self.model_route.reset()
        while True:
            ob_signal=ob[:17]
            ob_route=ob[17:]

            policy_signal = self.model_signal.forward(ob_signal, done, 'p')
            policy_route=self.model_route.forward(ob_route, done, 'p')
            policy=policy_signal+policy_route

            self.env.update_fingerprint(policy)
            action_signal = []
            for pi in policy_signal:
                if policy_type != 'deterministic':
                    action_signal.append(np.random.choice(np.arange(len(pi)), p=pi))
                else:
                    action_signal.append(np.argmax(np.array(pi)))

            action_route = []
            for pi in policy_route:
                if policy_type != 'deterministic':
                    action_route.append(np.random.choice(np.arange(len(pi)), p=pi))
                else:
                    action_route.append(np.argmax(np.array(pi)))

            next_ob, done= self.env.step_separate(action_signal,action_route)

            if done:
                break
            ob=next_ob


## 12. Main Entry Point (`main.py`)

Set `option` to one of:

| `option` | Action |
|----------|--------|
| `'train'` | Train MA2C from scratch, save model + CSVs |
| `'evaluate'` | Load saved model, run full evaluation, write output CSVs |
| `'greedy'` | Run the `MaltaController` baseline, write `greedy_baseline.csv` |
| `'get_parameter'` | One free-run episode to collect state/reward statistics |

Other flags:
- `separate_train=True` — load signal and route models trained independently
- `diff=True` + `compliance`/`load` — sensitivity sweeps over CAV penetration rate or demand

**Before running:** make sure `Malta/data/malta.net.xml` exists and SUMO is on your PATH.


In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# Force CPU-only by default. Originally needed because the per-call
# Python<->GPU copy cost per agent dominated for batch-size-1 LSTM
# inference. After the IA2C.forward/backward batching refactor (one
# sess.run for all 190 agents), the GPU may actually win — try toggling
# this off (comment the next line) once a CPU-only run is verified.
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
# Enable Intel oneDNN CPU math kernels (typically 2-4x speedup on CPU
# matmul/conv/LSTM compared to the stock TF kernels).
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Let oneDNN/OpenMP use every core. os.cpu_count() returns logical cores.
_n_cores = str(os.cpu_count() or 8)
os.environ.setdefault('OMP_NUM_THREADS', _n_cores)
os.environ.setdefault('KMP_BLOCKTIME', '0')  # don't spin between calls
os.environ.setdefault('KMP_AFFINITY', 'granularity=fine,compact,1,0')

from tensorflow.python.util import deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False

import configparser
import tensorflow as tf
# Report what TF can see — useful to confirm oneDNN + GPU detection.
print('>>> TF version:', tf.__version__, '| logical CPUs:', _n_cores, flush=True)
try:
    _gpus = tf.config.list_physical_devices('GPU')
    print('>>> TF GPUs visible:', _gpus or 'none (CPU-only run)', flush=True)
except Exception as _e:
    print('>>> TF GPU probe failed:', _e, flush=True)
tf = tf.compat.v1
tf.disable_v2_behavior()
import threading

from envs.Malta_env import MaltaEnv as SiouxEnv, MaltaController as SiouxController
from agents.models import MA2C
from utils import *  

def init_env(config, port=0, naive_policy=False):
    if not naive_policy:
        return SiouxEnv(config, problem, port=port)
    else:
        env = SiouxEnv(config, problem, port=port)
        policy = SiouxController(env.node_names, env.rc_names)
        return env, policy


def get_parameter(scenario):
    config_dir = default_config_dir
    config = configparser.ConfigParser()
    config.read(config_dir)
    env = SiouxEnv(config['ENV_CONFIG'],problem, port=0)
    env.get_parameter(scenario)
    env.terminate()

def train():
    base_dir = default_base_dir
    dirs = init_dir(base_dir)
    store_dir=default_store_dir
    dirs2=init_dir(store_dir)
    init_log(dirs2['log'])
    config_dir = default_config_dir
    copy_file(config_dir, dirs['data'])
    config = configparser.ConfigParser()
    config.read(config_dir)
    test_mode='no_test'
    in_test, post_test = init_test_flag(test_mode)

    # init env
    print(">>> Initializing env (launches SUMO for dimensions)...", flush=True)
    env = init_env(config['ENV_CONFIG'], port=0)
    logging.info('Training: s dim: %d, a dim %d, s dim ls: %r, a dim ls: %r' %
                 (env.n_s, env.n_a, env.n_s_ls, env.n_a_ls))

    print(">>> Building MA2C model...", flush=True)

    # init step counter
    total_step = int(config.getfloat('TRAIN_CONFIG', 'total_step'))
    test_step = int(config.getfloat('TRAIN_CONFIG', 'test_interval'))
    log_step = int(config.getfloat('TRAIN_CONFIG', 'log_interval'))
    global_counter = Counter(total_step, test_step, log_step)

    
    # init centralized or multi agent
    seed = config.getint('ENV_CONFIG', 'seed')

    if problem == 'signal_route':
        model = MA2C(env.n_s_ls, env.n_a_ls, env.n_w_ls, env.n_f_ls, total_step,
                     config['MODEL_CONFIG'], problem, agent_type=env.agent_type, n_o_ls=env.n_o_ls, seed=seed)
    else:
        model = MA2C(env.n_s_ls, env.n_a_ls, env.n_w_ls, env.n_f_ls, total_step,
                     config['MODEL_CONFIG'], problem, seed=seed)


    summary_writer =tf.summary.FileWriter(dirs2['log'])
    trainer = Trainer(env, model,problem, global_counter, summary_writer, in_test, output_path=dirs['data'])
    print(">>> Starting trainer.run() — first episode loading SUMO...", flush=True)
    trainer.run()

    # post-training test
    if post_test:
        tester = Tester(env, model, problem,global_counter, summary_writer, dirs['data'])
        tester.run_offline(dirs['data'])

    # save model
    final_step = global_counter.cur_step
    logging.info('Training: save final model at step %d ...' % final_step)
    model.save(dirs2['model'], final_step)


def evaluate_fn(agent, output_dir, seeds, port, demo, policy_type):
    # load config file for env
    config_dir = default_config_dir
    config = configparser.ConfigParser()
    config.read(config_dir)

    # init env
    env, greedy_policy = init_env(config['ENV_CONFIG'], port=port, naive_policy=True)
    logging.info('Evaluation: s dim: %d, a dim %d, s dim ls: %r, a dim ls: %r' %
                 (env.n_s, env.n_a, env.n_s_ls, env.n_a_ls))
    env.init_test_seeds(seeds)
    if separate_train==True:
        n_s_ls=[46, 36, 14, 22, 38, 38, 30, 22, 28, 34, 28, 30, 12, 14, 14, 28, 24]
        n_a_ls=[5, 4, 2, 2, 4, 4, 2, 2, 2, 4, 2, 4, 2, 2, 2, 4, 2]
        n_w_ls=[10, 8, 6, 6, 8, 8, 6, 6, 6, 8, 6, 8, 6, 6, 6, 8, 6]
        n_f_ls=[11, 7, 3, 6, 9, 9, 8, 6, 7, 6, 7, 7, 1, 3, 3, 5, 7]
        model_signal = MA2C(n_s_ls, n_a_ls, n_w_ls, n_f_ls, 0,
                             config['MODEL_CONFIG'], 'signal', agent_type=env.agent_type, n_o_ls=env.n_o_ls)
        n_s_ls=[1, 1, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1]
        n_a_ls=[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
        n_w_ls=[6, 7, 10, 7, 6, 10, 7, 9, 7, 10, 6, 9]
        n_f_ls=[0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0]
        model_route=MA2C(n_s_ls, n_a_ls, n_w_ls, n_f_ls, 0,
                             config['MODEL_CONFIG'], 'route', agent_type=env.agent_type, n_o_ls=env.n_o_ls)
        if not model_signal.load(default_base_dir+ '/signal/model/'):
            return
        if not model_route.load(default_base_dir+'/route/model/'):
            return
        env.agent=agent
        evaluator = Evaluator_separate(env, model_signal,model_route, output_dir, demo=demo, policy_type=policy_type)
        evaluator.run()
        return

    if problem == 'signal_route':
        model = MA2C(env.n_s_ls, env.n_a_ls, env.n_w_ls, env.n_f_ls, 0,
                             config['MODEL_CONFIG'], problem, agent_type=env.agent_type, n_o_ls=env.n_o_ls)
    else:
        model = MA2C(env.n_s_ls, env.n_a_ls, env.n_w_ls, env.n_f_ls, 0,
                             config['MODEL_CONFIG'], problem)

    if not model.load(default_store_dir + '/model/'):
        return

    env.agent = agent
    evaluator = Evaluator(env, model, output_dir, demo=demo, policy_type=policy_type)
    if diff==False:
        evaluator.run()
    else:
        if compliance==True:
            evaluator.run_diff_com()
        elif load==True:
            evaluator.run_diff_load()


def run_greedy():
    """Run one full episode with the MaltaController (greedy/naive policy) and save metrics."""
    import pandas as pd

    config = configparser.ConfigParser()
    config.read(default_config_dir)
    env, policy = init_env(config['ENV_CONFIG'], port=0, naive_policy=True)

    store_dir = default_store_dir
    dirs = init_dir(store_dir, pathes=['greedy_data', 'log'])
    init_log(dirs['log'])

    seeds_str = config.get('ENV_CONFIG', 'test_seeds', fallback='10000,20000')
    seeds = [int(s) for s in seeds_str.split(',')]
    env.init_test_seeds(seeds)
    env.train_mode = False

    data = []
    for test_ind in range(env.test_num):
        ob = env.reset(gui=False, test_ind=test_ind)
        done = False
        total_reward = 0
        total_arrived = 0
        total_departed = 0
        total_wait = 0
        total_travel = 0
        signal_n_a = env.n_a_ls[:len(env.node_names)]
        step_count = 0
        while not done:
            actions = policy.forward(ob)
            # clip each signal action to its node's valid phase range
            actions = [min(a, n - 1) for a, n in zip(actions, signal_n_a)]
            # route agents get a fixed action (stay on current route) when using greedy signal only
            full_actions = actions + [0] * len(env.rc_names)
            ob, reward, done, global_reward, arrived, departed, wait, travel, _ = env.step(full_actions)
            step_count += 1
            if step_count % 72 == 0:
                print('  episode %d | sim time %ds / %ds | arrived %d' % (
                    test_ind, step_count * 5, 3600, total_arrived), flush=True)
            total_reward += global_reward
            total_arrived += arrived
            total_departed += departed
            total_wait += wait
            total_travel += travel

        env.terminate()
        if total_departed > 0:
            avg_wait = total_wait / total_departed
            avg_travel = total_travel / total_departed
        else:
            avg_wait = avg_travel = float('nan')

        log = {
            'agent': 'greedy',
            'test_id': test_ind,
            'reward': total_reward,
            'total arrived': total_arrived,
            'total departed': total_departed,
            'average waiting time': avg_wait,
            'average travel time': avg_travel,
        }
        data.append(log)
        logging.info('Greedy test %d: avg_wait=%.2f s, avg_travel=%.2f s' % (test_ind, avg_wait, avg_travel))

    out_path = os.path.join(dirs['greedy_data'], 'greedy_baseline.csv')
    pd.DataFrame(data).to_csv(out_path, index=False)
    logging.info('Greedy baseline saved to %s' % out_path)


def evaluate():
    store_dir = default_store_dir
    dirs = init_dir(store_dir, pathes=['eva_data', 'eva_log'])
    init_log(dirs['eva_log'])
    agents={'ma2c'}
    seeds = ','.join([str(i) for i in range(10000, 100001, 10000)])
    policy_type = 'default'
    logging.info('Evaluation: policy type: %s, random seeds: %s' % (policy_type, seeds))
    if not seeds:
        seeds = []
    else:
        seeds = [int(s) for s in seeds.split(',')]
    threads = []
    for i, agent in enumerate(agents):
        demo=False
        thread = threading.Thread(target=evaluate_fn,
                                  args=(agent, dirs['eva_data'], seeds, i, demo, policy_type))
        thread.start()
        threads.append(thread)
    for thread in threads:
        thread.join()


default_config_dir = './config/config_ma2c_Malta.ini'
scenario = 'Malta'

option='train_then_evaluate' # 'train', 'evaluate', 'train_then_evaluate', 'greedy', 'get_parameter'

separate_train=False

diff=False
compliance=False
load=False

problem='signal_route'
default_base_dir = scenario
if separate_train==True:
    default_store_dir = default_base_dir + '/signal_route(separate_training)'
else:
    default_store_dir=default_base_dir+ '/' + problem


if __name__ == '__main__':
    if option == 'train':
        train()
    elif option == 'train_then_evaluate':
        print(">>> Stage 1/2: training...", flush=True)
        train()
        print(">>> Stage 2/2: evaluating freshly-trained model...", flush=True)
        evaluate()
        print(">>> train_then_evaluate complete.", flush=True)
    elif option == 'evaluate':
        evaluate()
    elif option == 'greedy':
        run_greedy()
    elif option == 'get_parameter':
        get_parameter(scenario)


>>> TF version: 2.21.0 | logical CPUs: 12
>>> TF GPUs visible: none (CPU-only run)
>>> Stage 1/2: training...
>>> Initializing env (launches SUMO for dimensions)...


[Malta route fix] corridor 1 alt 1: replaced ['54444410#2', '1264291443', '33698427#1'] with 9-edge path
[Malta route fix] corridor 1 alt 2: replaced ['824060363#1'] with 10-edge path
[Malta route fix] corridor 4 alt 1: replaced ['-743322159'] with 16-edge path
[Malta route fix] corridor 4 alt 2: replaced ['743322159'] with 12-edge path
[Malta detectors] emitted 442 laneAreaDetectors (TL-incoming lanes only)


2026-05-21 16:13:17,407 [INFO] Skipping TL node not in phase_node_map: 2385491667
2026-05-21 16:13:17,408 [INFO] Skipping TL node not in phase_node_map: 6765843590
2026-05-21 16:13:19,800 [INFO] Training: s dim: 979, a dim 6, s dim ls: [2, 4, 2, 2, 2, 2, 2, 1, 3, 2, 2, 2, 2, 2, 2, 4, 2, 4, 1, 2, 4, 2, 4, 2, 2, 2, 2, 4, 3, 4, 4, 1, 2, 2, 4, 2, 2, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 2, 5, 5, 2, 3, 4, 2, 2, 2, 4, 2, 5, 2, 4, 2, 3, 2, 4, 4, 2, 2, 1, 2, 3, 1, 2, 2, 3, 4, 2, 2, 2, 2, 2, 4, 4, 2, 4, 2, 2, 2, 2, 2, 2, 2, 5, 2, 8, 8, 1, 2, 2, 2, 2, 4, 1, 1, 2, 2, 1, 2, 2, 4, 2, 2, 2, 2, 2, 4, 4, 1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 4, 2, 2, 2, 2, 4, 8, 2, 2, 2, 2, 2, 2, 6, 2, 4, 2, 1, 2, 3, 8, 4, 2, 2, 2, 2, 2, 1, 4, 5, 2, 1, 3, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 1, 4, 2, 2, 3, 2, 2, 1, 1, 1, 1], a dim ls: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 4, 4, 3, 3

>>> Building MA2C model...



2026-05-21 16:18:57,223 [WARNING] From C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook\agents\models.py:409: The name tf.train.Saver is deprecated. Please use tf.compat.v1.train.Saver instead.



2026-05-21 16:19:03,275 [WARNING] From C:\Users\kylea\Documents\Mcast\Mcast 6.2B\Thesis notebook\Thesis notebook\agents\policies.py:76: The name tf.summary.merge is deprecated. Please use tf.compat.v1.summary.merge instead.



## 13. Results Visualisation

Load the saved CSVs and reproduce the training and evaluation plots.
No SUMO required for this section.


In [ ]:
%matplotlib inline
import os, pandas as pd, matplotlib.pyplot as plt

# Training curve
train_csv = "Malta/signal_route/train_reward.csv"
if os.path.exists(train_csv):
    df = pd.read_csv(train_csv)
    train_df = df[df["test_id"] == -1].copy()
    metrics = ["reward", "average waiting time", "average travel time"]
    ylabels = ["Cumulative Reward", "Avg Waiting Time (s)", "Avg Travel Time (s)"]
    fig, axes = plt.subplots(len(metrics), 1, figsize=(10, 4 * len(metrics)))
    for ax, metric, ylabel in zip(axes, metrics, ylabels):
        ax.plot(train_df["step"], train_df[metric], linewidth=1.5, color="steelblue")
        ax.set_xlabel("Training Step"); ax.set_ylabel(ylabel); ax.set_title(ylabel)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_curve.png", dpi=150)
    plt.show()
    print("Training curve saved to training_curve.png")
else:
    print(f"File not found: {train_csv}")


In [ ]:
%matplotlib inline
# MA2C vs Greedy bar chart
greedy_csv = "Malta/signal_route/greedy_data/greedy_baseline.csv"
agent_csv  = "Malta/signal_route/train_reward.csv"
records = []
if os.path.exists(agent_csv):
    df = pd.read_csv(agent_csv)
    test_df = df[df["test_id"] >= 0]
    if not test_df.empty:
        records.append({"label": "MA2C", "avg_wait": test_df["average waiting time"].mean(), "avg_travel": test_df["average travel time"].mean()})
if os.path.exists(greedy_csv):
    gdf = pd.read_csv(greedy_csv)
    if "average waiting time" in gdf.columns:
        records.append({"label": "Greedy", "avg_wait": gdf["average waiting time"].mean(), "avg_travel": gdf["average travel time"].mean()})
if records:
    labs = [r["label"] for r in records]
    colors = ["steelblue", "coral"]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, key, title in zip(axes, ["avg_wait", "avg_travel"], ["Avg Waiting Time (s)", "Avg Travel Time (s)"]):
        vals = [r[key] for r in records]
        bars = ax.bar(labs, vals, color=colors[:len(vals)])
        ax.set_title(title); ax.set_ylabel("Seconds"); ax.grid(True, axis="y", alpha=0.3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig("policy_comparison.png", dpi=150)
    plt.show()
    print("Comparison chart saved to policy_comparison.png")
else:
    print("No result CSVs found - run training / greedy first.")


In [ ]:
%matplotlib inline
# Per-agent reward breakdown
agent_reward_csv = "Malta/signal_route/agent_reward.csv"
if os.path.exists(agent_reward_csv):
    ar = pd.read_csv(agent_reward_csv, header=None)
    print(f"Per-agent reward matrix shape: {ar.shape}  (episodes x agents)")
    ax = ar.mean(axis=0).plot(kind="bar", figsize=(14, 4), color="steelblue", title="Mean Cumulative Reward per Agent (across all training episodes)")
    ax.set_xlabel("Agent index"); ax.set_ylabel("Cumulative reward")
    plt.tight_layout()
    plt.savefig("agent_rewards.png", dpi=150)
    plt.show()
    print("Saved to agent_rewards.png")
else:
    print(f"File not found: {agent_reward_csv}")
